# Notebook 02 — Beat Averaging and Clustering Across All Animals

**Project:** Reproducible Signal Processing and Multivariate Analysis of Preclinical Mouse ECG Data for Cardiotoxicity Detection  
**Author:** Vishvasundar (MSc, University College Cork)  
**MSc thesis project**

## What this notebook does

Process every `.txt` ECG export in `data/`, isolate each animal's clean baseline window, detect R peaks, build an averaged-beat template, extract morphology features, and cluster the animals. Final outputs: `outputs/all_animals_features.csv` and four PNG figures.

### Annotation rule (Fix 1)
Annotations are matched generously by *prefix character* (`#*`, `#3`, `#1`) and the **gap between paired markers must be 10–120 s**. If the only candidate pair is too long (e.g. a 15-minute baseline), the first 60 s after the start marker is used. If only a single marker exists, a 30 s window after it is used. Non-baseline procedure markers (containing `oc`, `jugular`, `jugcan`) are excluded.

### ECG column rule (Fix 2)
The LabChart header tells us how many channels exist and which is *Channel 3*. The number of leading time columns is computed dynamically from `n_cols_in_first_data_row − n_channels`. Channel 3 is always the ECG. A sanity check rejects any column whose values look like BPM (300–700) instead of mV (±15).

### Filtering — Adaptive Physiology-Driven Framework
Butterworth bandpass **0.5–150 Hz** (zero-phase `filtfilt`). The high cutoff is raised from the human-ECG-standard 40 Hz to **150 Hz** because the mouse QRS is only ~10 ms wide and carries spectral energy well above 40 Hz; 150 Hz preserves the sharp QRS/J-wave morphology and is safely below the 500 Hz Nyquist (1 kHz sampling).

### Robust R-peak detection (Fix 3)
`height = median + 4 × MAD` (median absolute deviation), so big artifact spikes do not raise the threshold above the real ~0.2 mV R peaks; `prominence = max(0.3 × MAD, 0.005 mV)`. The refractory **distance = `REFRACTORY_MS` (55 ms)** — the absolute physiological minimum between successive R peaks (matches Notebook 01); `MIN_RR_MS` (75 ms) is kept as a physiological RR-plausibility floor. If the resulting heart rate is outside 300–700 bpm we retry on the inverted signal. Files still outside 300–700 bpm are flagged `NEEDS_REVIEW` and kept in the audit list.

### QTc formula
**Mitchell:** `QTc = QT / √(RR/100)`, with QT and RR in milliseconds. This is the mouse-specific correction used in this pipeline. The output CSV carries an explicit `qtc_formula` column (value `Mitchell`) documenting which formula was applied.

> ⚠️ **OPEN ISSUE — QT measurement method differs between Notebook 01 and Notebook 02 (do not resolve yet; needs review).**
>
> The **QTc formula is identical** in both notebooks, but the **QT input differs** because of how each notebook measures the QT interval:
> - **Notebook 01:** QT measured **per beat** on the raw filtered baseline signal, then averaged.
> - **Notebook 02 (this notebook):** QT measured on the **4-beat averaged template** (return-to-baseline within a fixed window).
>
> Same animal, same formula — the difference is purely the QT measurement method. Robust QT-endpoint detection in mouse ECG is genuinely hard (T wave merges with the J wave, no clean isoelectric return), so this is **deliberately left unresolved** pending review. Do **not** change the QT method until that review has happened.

### Scale (Fix 4)
No animal IDs are hardcoded. The pipeline iterates every `*.txt` file in the data folder. Each file is wrapped in `try/except` so one bad file cannot crash the run.

## Setup — imports and project paths

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks, iirnotch
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, fcluster

from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR    = PROJECT_ROOT / 'data'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

FS                  = 1000
BANDPASS_LOW_HZ     = 0.5
BANDPASS_HIGH_HZ    = 150.0      # Adaptive Physiology-Driven Framework: raised 40 -> 150 Hz
BANDPASS_ORDER      = 2

# --- R-peak timing (Adaptive Physiology-Driven Framework) -----------------
REFRACTORY_MS       = 55
MIN_RR_MS           = 75
BEATS_TO_AVERAGE    = 4
PRE_R_MS            = 100
POST_R_MS           = 150

HR_NORMAL_LOW       = 400        # green/red on the bar chart
HR_NORMAL_HIGH      = 500
HR_ACCEPT_LOW       = 300        # pipeline pass/fail gate — NEEDS_REVIEW trigger
HR_ACCEPT_HIGH      = 700

GAP_MIN_S           = 10
GAP_MAX_S           = 120
GAP_MAX_LONG_S      = 900
LONG_USE_S          = 60
SINGLE_FALLBACK_S   = 30

EXCLUDE_KEYWORDS    = ("oc 1", "oc 2", "oc1", "oc2",
                       "jugular", "jugcan", "injection", "inject")
END_KEYWORDS        = ("end", "done", "stop")
START_HINTS         = ("baseline", "ecg", "start")

# --- Window quality guard (BUG 2 fix) ------------------------------------
# Gate is [350, 700] — documented mouse physiological range, consistent with
# run_recording_quality_scan.py.  Distinct from HR_ACCEPT_LOW/HIGH ([300,700])
# which is the pipeline pass/fail threshold; the quality guard is stricter and
# only used when choosing a clean sub-window inside a NEEDS_REVIEW recording.
QUALITY_WIN_S        = 15    # sub-window width (seconds)
QUALITY_STEP_S       =  5    # slider step (seconds)
QUALITY_SEARCH_MAX_S = 90    # max search distance past annotation start (seconds)
QUALITY_HR_LOW       = 350   # quality gate lower HR bound (bpm)
QUALITY_HR_HIGH      = 700   # quality gate upper HR bound (bpm)
QUALITY_RRSTD_MAX    = 120   # rr_std ceiling for a "clean" window (ms)
QUALITY_MIN_BEATS    = 10    # minimum peaks required in a quality window

print(f"Data folder   : {DATA_DIR}")
print(f"Output folder : {OUTPUTS_DIR}")
print(f"Figure folder : {FIGURES_DIR}")


## Helper functions

In [ ]:
# --- Filename --------------------------------------------------------------
def parse_filename(path: Path):
    """Return (animal_id, recording_date). Handles 4-group names
       (year, month, day, animal) and 3-group names (year, month, animal).
       Repairs year tokens longer than 4 digits (e.g. '22024' -> '2024')
       by taking the last 4 digits of the token.
       Prints an explicit warning if parsing still fails so no file is
       ever dropped silently."""
    nums = re.findall(r"\d+", path.stem)
    try:
        if len(nums) >= 4:
            y_str, m, d, a = nums[0], int(nums[1]), int(nums[2]), int(nums[3])
            y = int(y_str[-4:]) if len(y_str) > 4 else int(y_str)
            return a, pd.Timestamp(year=y, month=m, day=d)
        if len(nums) == 3:
            y_str, m, a = nums[0], int(nums[1]), int(nums[2])
            y = int(y_str[-4:]) if len(y_str) > 4 else int(y_str)
            return a, pd.Timestamp(year=y, month=m, day=1)
    except (ValueError, TypeError) as exc:
        print(f"  WARNING parse_filename: cannot parse '{path.name}' — {exc}")
    return None, None


# --- Header / column layout (Fix 2) ---------------------------------------
def parse_header_and_layout(path: Path):
    """Read header lines, then the first data row, and figure out:
         n_header   : number of header lines
         lead_cols  : number of leading time/date columns
         ecg_col    : 0-based column index of Channel 3 (the ECG mV channel)
       Lead cols are computed as (n_columns_in_first_data_row - n_channels_in_header)."""
    info = {}
    n_header = 0
    first_data_row = None
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            s = line.lstrip()
            if not s or s[0].isdigit() or s[0] in "+-.":
                first_data_row = line.rstrip("\n")
                break
            n_header += 1
            if "=" in line:
                key, _, rest = line.partition("=")
                info[key.strip()] = rest.strip("\n").strip("\t").split("\t")

    titles = [t.strip() for t in info.get("ChannelTitle", []) if t.strip()]
    n_channels = max(len(titles), 1)
    n_cols_first_row = len(first_data_row.split("\t")) if first_data_row else (n_channels + 1)
    lead_cols = max(1, n_cols_first_row - n_channels)

    # Find Channel 3 in the channel-title list; fall back to first/only channel.
    ch3_idx = None
    for i, t in enumerate(titles):
        if t.lower() == "channel 3":
            ch3_idx = i
            break
    if ch3_idx is None:
        ch3_idx = 0
    ecg_col = lead_cols + ch3_idx
    return n_header, lead_cols, ecg_col, titles


# --- Marker detection ------------------------------------------------------
# NOTE: no word boundary (\b) — '*' is non-word, so '\b' would never match #*.
# This matches '#*', '#1', '#2', '#3'.
_MARKER_RE = re.compile(r"#([*1-3])")

def _looks_like_end(text: str)        -> bool:
    low = text.lower()
    return any(k in low for k in END_KEYWORDS)

def _looks_like_excluded(text: str)   -> bool:
    low = text.lower()
    return any(k in low for k in EXCLUDE_KEYWORDS)

def _looks_like_baseline(text):
    return "baseline" in text.lower().replace("basline", "baseline")


# --- File loader -----------------------------------------------------------
def load_ecg_file(path: Path, n_header: int, ecg_col: int):
    """Stream the file. Return:
         voltage     : 1-D float mV array
         markers     : list of (row_index, full_annotation_text, prefix_char)
                       where prefix_char is '*', '1', '2', or '3'."""
    voltages = []
    markers  = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for _ in range(n_header):
            f.readline()
        for line in f:
            stripped = line.rstrip("\n")
            parts = stripped.split("\t")
            if len(parts) <= ecg_col:
                continue
            try:
                v = float(parts[ecg_col])
            except ValueError:
                v = np.nan
            voltages.append(v)
            m = _MARKER_RE.search(stripped)
            if m:
                prefix = m.group(1)              # '*' / '1' / '2' / '3'
                ann_text = stripped[m.start():].strip()
                markers.append((len(voltages) - 1, ann_text, prefix))

    voltage = np.asarray(voltages, dtype=float)
    if np.isnan(voltage).any():
        idx = np.arange(len(voltage))
        good = ~np.isnan(voltage)
        if good.any():
            voltage = np.interp(idx, idx[good], voltage[good])
    return voltage, markers


# --- Baseline window (Fix 1) ----------------------------------------------
def find_baseline_window(markers, n_samples, fs=FS):
    """Tiered search for a baseline (start, end, rule_used, start_text, end_text).
       Tier A: same-prefix pair, gap 10-120 s.
       Tier B: cross-prefix pair, gap 10-120 s.
       Tier C: any pair whose texts both contain 'baseline', gap 10-900 s
               -> use only the first 60 s of that window.
       Tier D: any single start-like marker -> use a 30 s window after it.
       Markers whose text contains procedure keywords (oc, jugular...) are excluded."""
    MIN_GAP   = GAP_MIN_S      * fs
    MAX_GAP   = GAP_MAX_S      * fs
    LONG_GAP  = GAP_MAX_LONG_S * fs
    LONG_USE  = LONG_USE_S     * fs
    FALLBACK  = SINGLE_FALLBACK_S * fs

    tagged = []
    for idx, text, prefix in markers:
        if _looks_like_excluded(text):
            continue
        end_like = _looks_like_end(text)
        tagged.append({"idx": idx, "text": text, "prefix": prefix, "end": end_like})

    starts = [m for m in tagged if not m["end"]]
    ends   = [m for m in tagged if m["end"]]

    def _try(_starts, _ends, lo, hi, label):
        for s in _starts:
            for e in _ends:
                if e["idx"] <= s["idx"]:
                    continue
                gap = e["idx"] - s["idx"]
                if lo <= gap <= hi:
                    return s, e, label
        return None

    # Tier A — same-prefix, short window. '*' first because it's the most common.
    for pre in ("*", "1", "2", "3"):
        result = _try(
            [m for m in starts if m["prefix"] == pre],
            [m for m in ends   if m["prefix"] == pre],
            MIN_GAP, MAX_GAP, f"A_same_{pre}")
        if result:
            s, e, label = result
            return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier B — cross-prefix, short window
    result = _try(starts, ends, MIN_GAP, MAX_GAP, "B_cross")
    if result:
        s, e, label = result
        return s["idx"], min(e["idx"], n_samples), label, s["text"], e["text"]

    # Tier C — long baseline window (both markers contain 'baseline')
    bl_starts = [m for m in starts if _looks_like_baseline(m["text"])]
    bl_ends   = [m for m in ends   if _looks_like_baseline(m["text"])]
    result = _try(bl_starts, bl_ends, MIN_GAP, LONG_GAP, "C_long_baseline")
    if result:
        s, e, label = result
        end_idx = min(s["idx"] + LONG_USE, e["idx"], n_samples)
        return s["idx"], end_idx, label, s["text"], e["text"]

    # Tier D — single start-like marker, 30 s fallback
    candidate = None
    for m in starts:
        if any(k in m["text"].lower() for k in START_HINTS):
            candidate = m
            break
    if candidate is None and starts:
        candidate = starts[0]
    if candidate is None and tagged:
        candidate = tagged[0]
    if candidate is not None:
        end_idx = min(candidate["idx"] + FALLBACK, n_samples)
        return candidate["idx"], end_idx, "D_single_30s", candidate["text"], None

    return None, None, "no_markers", None, None


# --- Signal processing ----------------------------------------------------
def _remove_mains(x, fs, freqs=(50.0, 100.0), win_s=1.0, overlap=0.5):
    """Least-squares sinusoidal subtraction of mains tones (spectrum-fit).
    Removes only the 50/100 Hz sinusoid, preserving cardiac energy in that band
    (unlike a notch, which removes a frequency band). Windowed with a Hann
    cross-fade to track slow mains drift. See FILTERING_METHODS doc."""
    x = np.asarray(x, float); n = len(x); w = int(round(win_s * fs))
    def fit_sub(seg, t0):
        t = (np.arange(len(seg)) + t0) / fs
        D = np.column_stack([np.cos(2*np.pi*f*t) for f in freqs] +
                            [np.sin(2*np.pi*f*t) for f in freqs])
        coef, *_ = np.linalg.lstsq(D, seg, rcond=None)
        return seg - D @ coef
    if w >= n:
        return fit_sub(x, 0)
    step = max(1, int(round(w * (1 - overlap))))
    y = np.zeros(n); wsum = np.zeros(n); taper = np.hanning(w)
    for start in range(0, n - 1, step):
        end = min(start + w, n); tap = taper[:end - start]
        y[start:end] += fit_sub(x[start:end], start) * tap
        wsum[start:end] += tap
    wsum[wsum == 0] = 1.0
    return y / wsum


def bandpass_filter(x, fs=FS, low=BANDPASS_LOW_HZ, high=BANDPASS_HIGH_HZ, order=BANDPASS_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    y = filtfilt(b, a, x)
    # Mains removal by least-squares sinusoidal subtraction (spectrum-fit),
    # replacing the notch. Removes only the 50/100 Hz tone, preserving cardiac
    # energy in that band; cohort R-amplitude loss ~0.4% vs 1.6% for a Q=60 notch.
    y = _remove_mains(y, fs, freqs=(50.0, 100.0))
    return y


def _peaks_with_mad(sig, fs=FS, refractory_ms=REFRACTORY_MS):
    """Median + 4*MAD adaptive threshold (Adaptive Physiology-Driven Framework).
       Refractory distance = REFRACTORY_MS (55 ms): the absolute physiological
       minimum between successive R peaks (matches Notebook 01)."""
    if sig.size == 0:
        return np.array([], dtype=int), np.nan, np.nan
    med = float(np.median(sig))
    mad = float(np.median(np.abs(sig - med)))
    if mad <= 0:
        return np.array([], dtype=int), med, mad
    height     = med + 4 * mad
    prominence = max(0.3 * mad, 0.005)            # safety floor at 5 uV
    distance   = int(refractory_ms * fs / 1000)
    peaks, _   = find_peaks(sig, height=height, distance=distance, prominence=prominence)
    return peaks, height, prominence


def detect_r_peaks(sig, fs=FS):
    """Robust R-peak detection. Returns (peaks, inverted_flag, hr_bpm, threshold).
       1. MAD threshold on the signal as-is.
       2. If HR is outside 300-700 bpm, retry on the inverted signal.
       3. Whichever gives an HR closer to the 300-700 band wins."""
    def _hr(peaks):
        if len(peaks) < 2:
            return np.nan
        rr_ms = np.diff(peaks) * (1000.0 / fs)
        return 60000.0 / np.mean(rr_ms)

    pos_peaks, pos_h, _ = _peaks_with_mad(sig, fs)
    pos_hr = _hr(pos_peaks)
    pos_ok = HR_ACCEPT_LOW <= pos_hr <= HR_ACCEPT_HIGH if not np.isnan(pos_hr) else False
    if pos_ok:
        return pos_peaks, False, pos_hr, pos_h

    inv_peaks, inv_h, _ = _peaks_with_mad(-sig, fs)
    inv_hr = _hr(inv_peaks)
    inv_ok = HR_ACCEPT_LOW <= inv_hr <= HR_ACCEPT_HIGH if not np.isnan(inv_hr) else False
    if inv_ok:
        return inv_peaks, True, inv_hr, inv_h

    def _dist(hr):
        if np.isnan(hr):
            return np.inf
        if hr < HR_ACCEPT_LOW:
            return HR_ACCEPT_LOW - hr
        if hr > HR_ACCEPT_HIGH:
            return hr - HR_ACCEPT_HIGH
        return 0.0

    if _dist(inv_hr) < _dist(pos_hr):
        return inv_peaks, True, inv_hr, inv_h
    return pos_peaks, False, pos_hr, pos_h


def average_beats(sig, r_peaks, fs=FS,
                  pre_ms=PRE_R_MS, post_ms=POST_R_MS,
                  group_size=BEATS_TO_AVERAGE):
    pre  = int(pre_ms  * fs / 1000)
    post = int(post_ms * fs / 1000)
    win_len = pre + post
    t_ms = (np.arange(win_len) - pre) * (1000.0 / fs)
    beats = [sig[r - pre: r + post] for r in r_peaks
             if r - pre >= 0 and r + post <= len(sig)]
    if not beats:
        return None, t_ms, None
    beats = np.vstack(beats)                     # individual aligned beats (n_beats x win_len)
    n_full_groups = len(beats) // group_size
    if n_full_groups == 0:
        return beats.mean(axis=0), t_ms, beats
    grouped = beats[:n_full_groups * group_size].reshape(n_full_groups, group_size, win_len)
    return grouped.mean(axis=1).mean(axis=0), t_ms, beats


def _measure_one(sig, t_ms):
    """Measure R amplitude, QRS-FWHM, J wave, T wave, RT and QT on one beat
       (or the averaged template). Identical logic for template and per-beat."""
    r_idx = int(np.argmax(sig))
    r_peak = float(sig[r_idx])          # from-zero peak: FWHM + QT-tol reference only
    half = r_peak / 2.0
    left = r_idx
    while left > 0 and sig[left] > half:
        left -= 1
    right = r_idx
    while right < len(sig) - 1 and sig[right] > half:
        right += 1
    qrs_ms = float(t_ms[right] - t_ms[left])
    # BUG FIX (R-amplitude under-estimation): report R amplitude as the QRS
    # peak-to-peak deflection (R-to-S), not height above zero. The manual
    # M-cursor measures the full deflection; from-zero under-estimated by
    # ~31% mean vs manual across 6 validated animals (232,125,249,201,146,106).
    _qrs_w = (t_ms >= -15) & (t_ms <= 35)
    r_amp  = float(np.max(sig[_qrs_w]) - np.min(sig[_qrs_w])) if _qrs_w.any() else r_peak
    def window_max(s_ms, e_ms):
        mask = (t_ms >= s_ms) & (t_ms <= e_ms)
        if not mask.any():
            return np.nan, np.nan
        seg = sig[mask]; ts = t_ms[mask]
        k = int(np.argmax(seg))
        return float(seg[k]), float(ts[k])
    j_amp, _      = window_max(10, 30)
    t_amp, t_time = window_max(40, 80)
    rt_ms = float(t_time) if not np.isnan(t_time) else np.nan
    baseline = float(np.median(sig[t_ms < -70])) if (t_ms < -70).any() else 0.0

    # --- P / PR / Q / S extraction (UNVALIDATED: no manual ground-truth in the
    #     validation table; windows placed from clean-template morphology and
    #     mouse-ECG literature; report NaN when a wave is below the noise floor).
    _noise = float(np.std(sig[t_ms < -70])) if (t_ms < -70).any() else 0.0
    def _win(s_ms, e_ms):
        m = (t_ms >= s_ms) & (t_ms <= e_ms)
        return (sig[m] - baseline), t_ms[m], m.any()
    # P wave: positive atrial bump ahead of QRS (peak ~ -45 to -60 ms)
    _pseg, _pts, _pok = _win(-70, -35)
    p_amp = pr_ms = np.nan
    if _pok:
        _pk = int(np.argmax(_pseg))
        if _pseg[_pk] > max(0.008, 2.0 * _noise):          # noise-floor guard
            p_amp = float(_pseg[_pk])
            pr_ms = float(0.0 - _pts[_pk])                 # P-peak -> R-peak (t=0)
    # Q wave: small negative deflection immediately before R (often absent in mice)
    _qseg, _qts, _qok = _win(-14, -2)
    q_amp = float(np.min(_qseg)) if _qok else np.nan       # signed (<=0 expected)
    # S wave: negative trough just after R (~ +15 to +30 ms)
    _sseg, _sts, _sok = _win(8, 32)
    s_amp = float(np.min(_sseg)) if _sok else np.nan       # signed (<=0 expected)

    tol = 0.05 * r_peak
    qt_ms = np.nan
    for i in np.where(t_ms >= 40)[0]:
        if abs(sig[i] - baseline) <= tol:
            qt_ms = float(t_ms[i])
            break
    return r_amp, qrs_ms, j_amp, t_amp, rt_ms, qt_ms, p_amp, pr_ms, q_amp, s_amp


def _qt_per_beat(beat, t_ms, rr_ms=None):
    """Per-beat QT - RR-bounded and polarity-aware (BUG fix, Section 9).

       Fixes the 26-53% QT over-estimate. Root cause: the endpoint search ran
       past the T-wave and latched onto the FOLLOWING beat's P-wave, which in
       the averaged mouse beat is a LARGER deflection than the small/inverted
       true T-wave (verified on Animal 125: P-wave at ~0.68*RR, excursion
       ~0.05 mV vs true T ~0.03 mV). Two changes vs the Notebook-01 method:
         1. Polarity-aware T detection. The T-wave is found as the maximum
            baseline-relative EXCURSION |signal - baseline| (works on the
            inverted mouse T-wave), and the T-end is the first return of that
            excursion to T_END_FRAC of its own peak - a relative threshold,
            not a fixed 0.01 mV absolute crossing.
         2. RR-bounded search. The T search is capped at R + QT_MAX_FRAC*RR
            (0.5), a physiological QT ceiling for the mouse that sits safely
            below the next P-wave (~0.65-0.70 RR). When rr_ms is None it falls
            back to a fixed 100 ms window (old behaviour).

       Validated on Animal 125: mean QT 91 -> 46 ms (manual M-cursor 44 ms).
       Returns QT in ms if in [15, 90], else nan. R peak = sample t_ms == 0."""
    QT_MAX_FRAC = 0.5      # physiological QT ceiling as a fraction of RR
    T_END_FRAC  = 0.30     # excursion decay that marks the T-wave end
    n30  = int(30  * FS / 1000); n50 = int(50 * FS / 1000)
    n20  = int(20  * FS / 1000); n100 = int(100 * FS / 1000)
    r_idx = int(np.argmin(np.abs(t_ms)))

    # Pre-R baseline (R-50 .. R-20 ms), same window as the original method.
    lo = max(0, r_idx - n50); hi = max(lo + 1, r_idx - n20)
    baseline = float(np.mean(beat[lo:hi]))

    # T-search window: starts 30 ms after R (skips QRS + J-wave); ends at the
    # physiological QT ceiling 0.5*RR (or a fixed 100 ms if RR is unknown).
    # This cap is what keeps the search clear of the next beat's P-wave.
    s = r_idx + n30
    if rr_ms is not None and np.isfinite(rr_ms) and rr_ms > 0:
        search_end = min(len(beat), r_idx + int(QT_MAX_FRAC * rr_ms * FS / 1000))
    else:
        search_end = min(len(beat), r_idx + n100)
    if search_end <= s:
        return np.nan

    # Polarity-independent T peak: largest |signal - baseline| in the window.
    exc = np.abs(beat[s:search_end] - baseline)
    if exc.size == 0:
        return np.nan
    k = int(np.argmax(exc)); t_peak = s + k; peak_exc = float(exc[k])
    if peak_exc <= 0:
        return np.nan

    # T-end: first sample after the peak whose excursion decays below
    # T_END_FRAC of the peak excursion.
    tail = np.abs(beat[t_peak:search_end] - baseline)
    below = np.where(tail <= T_END_FRAC * peak_exc)[0]
    if len(below) == 0:
        return np.nan

    qt = (t_peak + int(below[0]) - r_idx) * 1000.0 / FS
    return float(qt) if 15 <= qt <= 90 else np.nan


def extract_morphology_features(template, t_ms, beats=None, rr_ms=None):
    """Morphology from the averaged template, plus per-beat standard deviations
       (qt_std_ms, qrs_std_ms, r_amplitude_std_mv) collected across individual beats."""
    keys = ["r_amplitude_mv", "qrs_duration_ms", "j_wave_amplitude_mv",
            "t_wave_amplitude_mv", "t_wave_excursion_mv", "rt_interval_ms", "qt_ms",
            "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv",
            "p_wave_amplitude_mv", "pr_interval_ms",
            "q_wave_amplitude_mv", "s_wave_amplitude_mv"]
    if template is None:
        return dict.fromkeys(keys, np.nan)

    (r_amp, qrs_ms, j_amp, t_amp, rt_ms, qt_template,
     p_amp, pr_ms, q_amp, s_amp) = _measure_one(template, t_ms)

    # Baseline-relative T-wave excursion (polarity-independent trigger for Rule B)
    _bl   = float(np.median(template[t_ms < -70])) if (t_ms < -70).any() else 0.0
    _tw_w = (t_ms >= 40) & (t_ms <= 80)
    t_wave_excursion = (float(np.max(np.abs(template[_tw_w] - _bl)))
                        if _tw_w.any() else np.nan)

    qt_ms = qt_template
    qt_std_ms = qrs_std_ms = r_amplitude_std_mv = np.nan
    if beats is not None and getattr(beats, "ndim", 0) == 2 and len(beats) > 1:
        r_amplitudes, qrs_durations, qt_estimates = [], [], []
        for b in beats:
            br, bqrs, _, _, _, _, _, _, _, _ = _measure_one(b, t_ms)
            r_amplitudes.append(br)
            qrs_durations.append(bqrs)
            bqt = _qt_per_beat(b, t_ms, rr_ms)
            if not np.isnan(bqt):
                qt_estimates.append(bqt)
        if len(qt_estimates) > 0:
            qt_ms = float(np.mean(qt_estimates))
        # Report R amplitude as the per-beat mean (consistent with qt_ms above);
        # the averaged template blunts the sharp R spike (~7% low, worst on 106).
        if len(r_amplitudes) > 0:
            r_amp = float(np.mean(r_amplitudes))
        qt_std_ms          = float(np.std(qt_estimates, ddof=1))  if len(qt_estimates)  > 1 else np.nan
        qrs_std_ms         = float(np.std(qrs_durations, ddof=1)) if len(qrs_durations) > 1 else np.nan
        r_amplitude_std_mv = float(np.std(r_amplitudes, ddof=1))  if len(r_amplitudes)  > 1 else np.nan

    return {
        "r_amplitude_mv":       r_amp,
        "qrs_duration_ms":      qrs_ms,
        "j_wave_amplitude_mv":  j_amp,
        "t_wave_amplitude_mv":  t_amp,
        "t_wave_excursion_mv":  t_wave_excursion,
        "rt_interval_ms":       rt_ms,
        "qt_ms":                qt_ms,
        "qt_std_ms":            qt_std_ms,
        "qrs_std_ms":           qrs_std_ms,
        "r_amplitude_std_mv":   r_amplitude_std_mv,
        "p_wave_amplitude_mv":  p_amp,
        "pr_interval_ms":       pr_ms,
        "q_wave_amplitude_mv":  q_amp,
        "s_wave_amplitude_mv":  s_amp,
    }


# --- Window quality guard (BUG 2 fix) ------------------------------------
def quality_window_search(voltage, annotation_start, n_samples, fs=FS):
    """Slide QUALITY_WIN_S sub-windows from annotation_start to find the
    cleanest segment closest to the annotation anchor.

    Gate: HR in [QUALITY_HR_LOW, QUALITY_HR_HIGH], rr_std <= QUALITY_RRSTD_MAX,
          n_beats >= QUALITY_MIN_BEATS.

    Scoring: among all passing windows, prefer the one closest to annotation_start
    (minimum movement); use rr_std as tiebreaker within the same 5s proximity bucket.

    quality_flag:
      'solid'                   — pass_rate >= 30% AND chosen rr_std <= p75
      'marginal_<n>/<tried>'    — best window found but fragile (low pass_rate
                                  OR chosen rr_std in top quartile); animal keeps
                                  NEEDS_REVIEW and is NOT promoted to OK
      'no_clean_window_found'   — zero windows passed the gate

    Always returns a dict (never None) so n_tried is always logged accurately."""
    WIN  = int(QUALITY_WIN_S  * fs)
    STEP = int(QUALITY_STEP_S * fs)
    search_end = min(n_samples, annotation_start + int(QUALITY_SEARCH_MAX_S * fs))

    candidates = []
    n_tried    = 0
    c = annotation_start
    while c + WIN <= search_end:
        n_tried += 1
        seg  = voltage[c:c + WIN]
        filt = bandpass_filter(seg)
        peaks, _, hr, _ = detect_r_peaks(filt)
        rr     = np.diff(peaks) * (1000.0 / fs)
        rr_std = float(np.std(rr, ddof=1)) if len(rr) > 1 else np.nan
        n_b    = len(peaks)
        ok = (QUALITY_HR_LOW <= hr <= QUALITY_HR_HIGH
              and not np.isnan(rr_std)
              and rr_std <= QUALITY_RRSTD_MAX
              and n_b >= QUALITY_MIN_BEATS)
        if ok:
            candidates.append({
                "start":   c,
                "hr":      float(hr),
                "rr_std":  rr_std,
                "n_beats": n_b,
                "dist":    c - annotation_start,
            })
        c += STEP

    n_passing = len(candidates)
    if n_passing == 0:
        return {
            "best_start":   None,
            "best_hr":      None,
            "best_rr_std":  None,
            "n_passing":    0,
            "n_tried":      n_tried,
            "quality_flag": "no_clean_window_found",
        }

    # Sort: closest to annotation_start first; rr_std breaks ties
    candidates.sort(key=lambda x: (x["dist"], x["rr_std"]))
    best = candidates[0]

    p75 = float(np.percentile([x["rr_std"] for x in candidates], 75))
    is_marginal = (n_passing / n_tried < 0.30) or (best["rr_std"] > p75)

    return {
        "best_start":   best["start"],
        "best_hr":      best["hr"],
        "best_rr_std":  best["rr_std"],
        "n_passing":    n_passing,
        "n_tried":      n_tried,
        "quality_flag": f"marginal_{n_passing}/{n_tried}" if is_marginal else "solid",
    }


## Part 1 — Process every file in the data folder

Every file is wrapped in `try/except`. Each one gets a status and goes into the per-file table at the end.

Status values:
- `OK` — heart rate in 300–700 bpm, used for clustering.
- `NEEDS_REVIEW` — beats were extracted but HR is outside 300–700 bpm. Kept in the audit list, not used for clustering.
- `FAILED_PEAKS` — markers found, segment valid, but fewer than 10 R peaks detected.
- `FAILED_ANNOTATION` — no usable markers in the file.
- `FAILED_COLUMN` — voltage column did not look like ECG (e.g. BPM values).
- `FAILED_FILE` — unexpected exception parsing the file.

In [ ]:
files = sorted(DATA_DIR.glob("*.txt"))
print(f"Found {len(files)} .txt files\n")

QTC_FORMULA_NAME = "Mitchell"

def mitchell_qtc(qt_ms, rr_ms):
    """QTc = QT / sqrt(RR / 100).  Returns NaN if inputs are invalid."""
    if np.isnan(qt_ms) or np.isnan(rr_ms) or rr_ms <= 0:
        return np.nan
    return float(qt_ms / np.sqrt(rr_ms / 100.0))

per_animal        = []     # OK animals only: templates, visualisations
all_animals       = []     # OK + NEEDS_REVIEW + FAILED_PEAKS — clustering input
templates         = {}     # animal_id -> (template, t_ms)
needs_review      = []     # rows with status NEEDS_REVIEW (kept for summary printing)
rows_all          = []     # one dict per file for the per-file table
window_audit_rows = []     # audit log for window quality guard

# NaN feature block for animals with no measurable morphology (FAILED_PEAKS)
_NAN_FEATS = {
    "r_amplitude_mv": np.nan, "qrs_duration_ms": np.nan,
    "j_wave_amplitude_mv": np.nan, "t_wave_amplitude_mv": np.nan, "t_wave_excursion_mv": np.nan,
    "rt_interval_ms": np.nan, "qt_ms": np.nan,
    "qtc_ms": np.nan, "qtc_formula": QTC_FORMULA_NAME,
    "qt_std_ms": np.nan, "qrs_std_ms": np.nan,
    "r_amplitude_std_mv": np.nan,
    "p_wave_amplitude_mv": np.nan, "pr_interval_ms": np.nan,
    "q_wave_amplitude_mv": np.nan, "s_wave_amplitude_mv": np.nan,
}

for path in files:
    row = {
        "file":           path.name,
        "animal_id":      None,
        "beats":          None,
        "HR_bpm":         None,
        "base_start":     None,
        "base_end":       None,
        "duration_s":     None,
        "rule_used":      None,
        "ecg_column":     None,
        "inverted":       False,
        "status":         "",
        "start_annotation": None,
        "end_annotation":   None,
    }
    rows_all.append(row)
    try:
        animal_id, rec_date = parse_filename(path)
        row["animal_id"] = animal_id
        if animal_id is None:
            row["status"] = "FAILED_FILE"
            row["rule_used"] = "filename parse"
            continue

        n_header, lead_cols, ecg_col, titles = parse_header_and_layout(path)
        row["ecg_column"] = ecg_col

        voltage, markers = load_ecg_file(path, n_header, ecg_col)
        if voltage.size == 0:
            row["status"]    = "FAILED_FILE"
            row["rule_used"] = "no voltage rows"
            continue

        med_abs = float(np.median(np.abs(voltage)))
        if med_abs > 50:
            row["status"]    = "FAILED_COLUMN"
            row["rule_used"] = f"voltage median |.| = {med_abs:.1f} (looks like BPM)"
            continue

        start, end, rule, s_text, e_text = find_baseline_window(markers, len(voltage))
        row["rule_used"]        = rule
        row["start_annotation"] = s_text
        row["end_annotation"]   = e_text
        if start is None:
            row["status"] = "FAILED_ANNOTATION"
            continue

        row["base_start"] = start
        row["base_end"]   = end
        row["duration_s"] = (end - start) / FS

        segment = voltage[start:end]
        if segment.size < 2 * FS:
            row["status"] = "FAILED_ANNOTATION"
            row["rule_used"] = f"{rule} (window too short: {segment.size} samples)"
            continue

        filtered = bandpass_filter(segment)
        peaks, inverted, hr, _ = detect_r_peaks(filtered)
        row["inverted"] = inverted
        if inverted:
            filtered = -filtered

        # --- Window quality guard (BUG 2 fix) --------------------------------
        # Trigger: HR outside pipeline acceptance band [300, 700].
        # Skipped for all currently-OK animals — zero change for clean files.
        # Gate inside search: [350, 700] (documented mouse range, stricter than pipeline).
        # Scoring: closest to annotation_start first; rr_std as tiebreaker.
        # Promotion rule: ONLY 'solid' windows are accepted. 'marginal' windows
        # keep the original annotation window and the animal stays NEEDS_REVIEW.
        _old_start    = start
        _old_hr       = float(hr) if not np.isnan(hr) else np.nan
        _window_moved = False

        if not (HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH):
            _qresult = quality_window_search(voltage, start, len(voltage))

            if _qresult["quality_flag"] == "solid":
                _new_start    = _qresult["best_start"]
                _window_moved = (_new_start != _old_start)
                start = _new_start
                end   = start + int(QUALITY_WIN_S * FS)
                row["base_start"] = start
                row["base_end"]   = end
                row["duration_s"] = QUALITY_WIN_S
                segment  = voltage[start:end]
                filtered = bandpass_filter(segment)
                peaks, inverted, hr, _ = detect_r_peaks(filtered)
                row["inverted"] = inverted
                if inverted:
                    filtered = -filtered
                action = "MOVED  " if _window_moved else "TRIMMED"
                print(f"  WINDOW {action} animal {animal_id}: "
                      f"{_old_start/FS:.1f}s -> {start/FS:.1f}s  "
                      f"HR {_old_hr:.1f} -> {hr:.1f} bpm  "
                      f"({_qresult['n_passing']}/{_qresult['n_tried']} passed, "
                      f"flag={_qresult['quality_flag']})")
            elif _qresult["quality_flag"].startswith("marginal"):
                # Fluky pass: best window found but fragile — do not promote.
                print(f"  WINDOW MARGINAL animal {animal_id}: "
                      f"best at {_qresult['best_start']/FS:.1f}s  "
                      f"HR {_qresult['best_hr']:.1f} bpm  "
                      f"({_qresult['n_passing']}/{_qresult['n_tried']} passed, "
                      f"rr_std={_qresult['best_rr_std']:.1f}) — keeping NEEDS_REVIEW")

            # Always log to audit; n_tried is always accurate (never falls back to 0).
            window_audit_rows.append({
                "animal_id":     animal_id,
                "file":          path.name,
                "old_start":     int(_old_start),
                "old_start_s":   round(_old_start / FS, 2),
                "new_start":     int(start),
                "new_start_s":   round(start / FS, 2),
                "window_moved":  _window_moved,
                "old_HR_bpm":    round(_old_hr, 2) if not np.isnan(_old_hr) else None,
                "new_HR_bpm":    round(float(hr), 2) if not np.isnan(hr) else None,
                "rr_std_chosen": round(_qresult["best_rr_std"], 2) if _qresult["best_rr_std"] is not None else None,
                "n_passing":     _qresult["n_passing"],
                "n_tried":       _qresult["n_tried"],
                "quality_flag":  _qresult["quality_flag"],
            })
        # --- End quality guard -----------------------------------------------

        if len(peaks) < 10:
            row["status"] = "FAILED_PEAKS"
            row["beats"]  = int(len(peaks))
            row["HR_bpm"] = float(hr) if not np.isnan(hr) else None
            # Include in clustering matrix with NaN morphology — missingness IS the signal
            all_animals.append({
                "animal_id":      animal_id,
                "recording_date": rec_date,
                "source_file":    path.name,
                "status":         "FAILED_PEAKS",
                "n_beats":        int(len(peaks)),
                "heart_rate_bpm": float(hr) if not np.isnan(hr) else np.nan,
                "rr_mean_ms":     np.nan,
                "rr_std_ms":      np.nan,
                "median_rr_ms":   np.nan,
                "pct_near_floor": np.nan,
                "rr_intervals_ms": np.array([], dtype=float),
                **_NAN_FEATS,
            })
            continue

        rr_ms      = np.diff(peaks) * (1000.0 / FS)
        template, t_ms, beats = average_beats(filtered, peaks)
        feats      = extract_morphology_features(template, t_ms, beats, rr_ms=float(np.median(rr_ms)))

        rr_mean_ms = float(np.mean(rr_ms))
        feats["qtc_ms"]      = mitchell_qtc(feats["qt_ms"], rr_mean_ms)
        feats["qtc_formula"] = QTC_FORMULA_NAME

        row["beats"]  = int(len(peaks))
        row["HR_bpm"] = float(hr)

        record = {
            "animal_id":      animal_id,
            "recording_date": rec_date,
            "source_file":    path.name,
            "n_beats":        int(len(peaks)),
            "heart_rate_bpm": float(hr),
            "rr_mean_ms":     rr_mean_ms,
            "rr_std_ms":      float(np.std(rr_ms, ddof=1)) if len(rr_ms) > 1 else 0.0,
            "median_rr_ms":   float(np.median(rr_ms)),
            "pct_near_floor": float(100.0 * np.mean(rr_ms <= 65.0)),
            "rr_intervals_ms": rr_ms,
            **feats,
        }

        if HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH:
            row["status"]    = "OK"
            record["status"] = "OK"
            per_animal.append(record)
            templates[animal_id] = (template, t_ms)
        else:
            row["status"]    = "NEEDS_REVIEW"
            record["status"] = "NEEDS_REVIEW"
            needs_review.append(record)
        all_animals.append(record)   # both OK and NEEDS_REVIEW enter the clustering matrix

    except Exception as e:
        row["status"]    = "FAILED_FILE"
        row["rule_used"] = f"{type(e).__name__}: {e}"

print(f"QTc formula     : {QTC_FORMULA_NAME}   [QTc = QT / sqrt(RR/100)]")
print(f"OK              : {sum(1 for r in rows_all if r['status']=='OK')}")
print(f"NEEDS_REVIEW    : {sum(1 for r in rows_all if r['status']=='NEEDS_REVIEW')}")
print(f"FAILED_PEAKS    : {sum(1 for r in rows_all if r['status']=='FAILED_PEAKS')}")
print(f"FAILED_ANNOTATION: {sum(1 for r in rows_all if r['status']=='FAILED_ANNOTATION')}")
print(f"FAILED_COLUMN   : {sum(1 for r in rows_all if r['status']=='FAILED_COLUMN')}")
print(f"FAILED_FILE     : {sum(1 for r in rows_all if r['status']=='FAILED_FILE')}")
print(f"\nClustering input (all_animals): {len(all_animals)} animals "
      f"(OK + NEEDS_REVIEW + FAILED_PEAKS)")

a201 = next((a for a in (per_animal + needs_review) if a["animal_id"] == 201), None)
if a201 is not None:
    print("\nQTc formula verification (Animal 201):")
    print(f"  QT = {a201['qt_ms']:.1f} ms,  RR = {a201['rr_mean_ms']:.1f} ms")
    print(f"      = {a201['qtc_ms']:.1f} ms")


### Per-file table (every input file, success or skip)

Saved to `outputs/per_file_audit.csv` for sorting and inspecting failures.

In [ ]:
audit_df = pd.DataFrame(rows_all)[[
    "animal_id", "file", "status", "beats", "HR_bpm",
    "base_start", "base_end", "duration_s",
    "rule_used", "ecg_column", "inverted",
    "start_annotation", "end_annotation",
]].copy()
audit_df = audit_df.sort_values(["status", "animal_id"], na_position="last").reset_index(drop=True)

audit_path = OUTPUTS_DIR / "per_file_audit.csv"
audit_df.to_csv(audit_path, index=False)
print(f"Saved per-file audit to: {audit_path}\n")

with pd.option_context("display.max_rows", 200,
                       "display.max_colwidth", 50,
                       "display.width", 220):
    print(audit_df.to_string(index=False))

# --- Window quality audit -------------------------------------------------
# One row per NEEDS_REVIEW animal (the only ones where the guard fires).
# window_moved=False means the guard ran but found the best window at the
# same start position as the annotation (end may still have been trimmed).
print("\n" + "=" * 70)
print("WINDOW QUALITY AUDIT")
print("=" * 70)
if window_audit_rows:
    wq_df = pd.DataFrame(window_audit_rows)[[
        "animal_id", "file",
        "old_start", "old_start_s", "new_start", "new_start_s",
        "window_moved",
        "old_HR_bpm", "new_HR_bpm",
        "rr_std_chosen", "n_passing", "n_tried",
        "quality_flag",
    ]].sort_values("animal_id").reset_index(drop=True)
    wq_path = OUTPUTS_DIR / "window_quality_audit.csv"
    wq_df.to_csv(wq_path, index=False)
    print(f"Saved -> {wq_path}\n")
    with pd.option_context("display.max_rows", 50, "display.width", 220):
        print(wq_df.to_string(index=False))
else:
    print("No animals triggered the window quality guard.")


### Recording dates — acute vs chronic

In [ ]:
if not all_animals:
    print("No animals processed successfully — cannot guess study groups.")
else:
    unique_dates = sorted({a["recording_date"].date() for a in all_animals
                           if a.get("recording_date") is not None})
    if len(unique_dates) >= 2:
        gaps = [(unique_dates[i + 1] - unique_dates[i]).days for i in range(len(unique_dates) - 1)]
        split_at = int(np.argmax(gaps))
        early_dates = set(unique_dates[: split_at + 1])
        study_label = {d: ("acute" if d in early_dates else "chronic") for d in unique_dates}
        print(f"Largest date gap = {max(gaps)} days -> splitting at {unique_dates[split_at]}")
    else:
        study_label = {d: "unknown" for d in unique_dates}
    for a in all_animals:
        rd = a.get("recording_date")
        a["study_guess"] = study_label.get(rd.date(), "unknown") if rd is not None else "unknown"
    grp_counts = pd.Series([a["study_guess"] for a in all_animals]).value_counts()
    print("\nAnimals per study group (all statuses):")
    print(grp_counts.to_string())


## Part 2 — Visualise all animals

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.tab20
ids_sorted = sorted(templates.keys())
show_legend = len(ids_sorted) <= 20
for i, aid in enumerate(ids_sorted):
    tmpl, t_ms = templates[aid]
    ax.plot(t_ms, tmpl, color=cmap(i % 20), linewidth=0.8,
            alpha=0.7, label=f"Animal {aid}" if show_legend else None)
ax.axvline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.5)
ax.set_xlabel("Time relative to R peak (ms)")
ax.set_ylabel("Voltage (mV)")
ax.set_title(f"Averaged Beat Template — {len(ids_sorted)} Animals")
if show_legend:
    ax.legend(loc="upper right", ncol=2, fontsize=8)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "07_all_averaged_beats.png", dpi=150)
plt.show()

In [ ]:
df_hr = pd.DataFrame(per_animal).sort_values("animal_id") if per_animal else pd.DataFrame()
if not df_hr.empty:
    colors = ["green" if HR_NORMAL_LOW <= hr <= HR_NORMAL_HIGH else "orange"
              for hr in df_hr["heart_rate_bpm"]]
    fig, ax = plt.subplots(figsize=(max(12, 0.18 * len(df_hr)), 5))
    ax.bar(df_hr["animal_id"].astype(str), df_hr["heart_rate_bpm"], color=colors, edgecolor="black")
    ax.axhline(HR_NORMAL_LOW,  color="gray", linestyle="--", linewidth=0.8)
    ax.axhline(HR_NORMAL_HIGH, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Animal ID")
    ax.set_ylabel("Heart rate (bpm)")
    ax.set_title(f"Heart Rate per Animal (green = {HR_NORMAL_LOW}-{HR_NORMAL_HIGH} bpm,"
                 f" orange = within 300-700 acceptance band)")
    ax.tick_params(axis="x", labelrotation=90)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "08_heart_rates.png", dpi=150)
    plt.show()
else:
    print("No animals to plot.")

In [ ]:
if per_animal:
    n = len(per_animal)
    ncols = 6 if n > 24 else 4
    nrows = max(1, int(np.ceil(n / ncols)))
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.2 * nrows), sharey=True)
    axes = np.array(axes).reshape(-1)
    sa = sorted(per_animal, key=lambda x: x["animal_id"])
    for i, a in enumerate(sa):
        ax = axes[i]
        ax.plot(a["rr_intervals_ms"], linewidth=0.5)
        ax.set_title(f"A{a['animal_id']} {a['heart_rate_bpm']:.0f}bpm", fontsize=7)
        ax.tick_params(labelsize=6)
        ax.grid(alpha=0.3)
    for j in range(len(sa), len(axes)):
        axes[j].axis("off")
    fig.suptitle("RR Intervals per Animal", y=1.01, fontsize=11)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "09_rr_all_animals.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No animals to plot.")

## Part 3 — Feature table

In [ ]:
feature_cols = [
    "animal_id", "recording_date", "study_guess", "source_file", "status",
    "n_beats", "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
    "median_rr_ms", "pct_near_floor",
    "r_amplitude_mv", "qrs_duration_ms",
    "j_wave_amplitude_mv", "t_wave_amplitude_mv", "t_wave_excursion_mv",
    "rt_interval_ms", "qt_ms", "qtc_ms",
    "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv",
    "p_wave_amplitude_mv", "pr_interval_ms",
    "q_wave_amplitude_mv", "s_wave_amplitude_mv",
    "qtc_formula",
]
# Build from all_animals (OK + NEEDS_REVIEW + FAILED_PEAKS); status column retained
df = pd.DataFrame([{k: a.get(k) for k in feature_cols} for a in all_animals])
if not df.empty:
    df = df.sort_values("animal_id").reset_index(drop=True)

# Before/after summary (Change 1)
print("=== Change 1: animals entering the feature table ===")
print(f"  BEFORE: 111 OK animals only")
if not df.empty and "status" in df.columns:
    counts = df["status"].value_counts()
    total  = len(df)
    print(f"  AFTER : {total} animals total")
    for st in ["OK", "NEEDS_REVIEW", "FAILED_PEAKS"]:
        n = int(counts.get(st, 0))
        delta = f" (+{n - (111 if st=='OK' else 0)})" if st == "OK" else f" (+{n})"
        print(f"    {st:<15}: {n}{delta}")
    print(f"\nNaN counts per feature (new rows carry structural NaN):")
    num_cols = [c for c in df.columns if df[c].dtype in (float, "float64") and c not in ("n_beats",)]
    nan_counts = df[num_cols].isna().sum()
    print(nan_counts[nan_counts > 0].to_string())
df.head(20) if not df.empty else "(empty)"


### Change 2 — Null-not-zero rules (applied by feature category)

Two physiological rules applied uniformly to all rows in `df` before the clustering matrix is built.
Neither rule touches signal-processing or feature-extraction logic — they operate only on the already-extracted columns.

**Rule A — Rhythm irregularity (RR-dependent features):**
If `rr_cv > RR_CV_IRREGULAR` (15 %), the mean RR is not clinically meaningful under an irregular rhythm; therefore `rr_mean_ms`, `qt_ms`, and `qtc_ms` (which all depend on a stable mean RR) are set to NaN.

**Rule B — T-wave isolation (repolarisation features):**
If `t_wave_amplitude_mv < T_WAVE_ISO_MIN_MV` (5 µV noise floor), the T-wave was not isolatable; therefore `t_wave_amplitude_mv`, `rt_interval_ms`, `qt_ms`, and `qtc_ms` are set to NaN.  Never 0.

In [ ]:
# Rhythm irregularity threshold: CV of RR > 15% means mean RR is not clinically
# meaningful -- a mean of a grossly non-stationary process tells you nothing.
RR_CV_IRREGULAR  = 0.15

# T-wave isolation floor (EXCURSION-based, not signed-peak-based):
# t_wave_excursion_mv = max|sig - baseline| over 40-80ms captures both
# upright and inverted T-waves.  Values below 5 uV are indistinguishable
# from baseline noise -- T-wave was not isolatable.
T_WAVE_ISO_MIN_MV = 0.005

if df.empty:
    print("df empty -- skipping null rules.")
else:
    # rr_cv must be computed before Rule A so the trigger value is preserved
    # even after rr_mean_ms is conditionally set to NaN.
    df["rr_cv"] = df["rr_std_ms"] / df["rr_mean_ms"]

    # --- Rule A: RR-dependent features ----------------------------------------
    irr_mask = df["rr_cv"] > RR_CV_IRREGULAR
    n_irr    = int(irr_mask.sum())
    rr_dep_cols = ["rr_mean_ms", "qt_ms", "qtc_ms"]
    before_irr  = df.loc[irr_mask, ["animal_id", "status", "rr_cv"] + rr_dep_cols].copy()
    df.loc[irr_mask, rr_dep_cols] = float("nan")

    print(f"Rule A -- rhythm irregularity (rr_cv > {RR_CV_IRREGULAR}): "
          f"{n_irr} animals -> {rr_dep_cols} set to NaN")
    if n_irr:
        print(before_irr.to_string(index=False))

    # --- Rule B: T-wave isolation (FIXED: excursion-based trigger) -----------
    # Old trigger: t_wave_amplitude_mv < T_WAVE_ISO_MIN_MV
    #   PROBLEM: for inverted T-waves, window_max(40-80ms) returns near-baseline
    #   (~0 mV), causing the rule to fire on normal animals.
    # New trigger: t_wave_excursion_mv < T_WAVE_ISO_MIN_MV
    #   t_wave_excursion_mv = max|sig - baseline| -- polarity-independent;
    #   fires only when the T-wave window is genuinely flat vs pre-R baseline.
    twave_mask = (df["t_wave_excursion_mv"].notna() &
                  (df["t_wave_excursion_mv"] < T_WAVE_ISO_MIN_MV))
    n_twave    = int(twave_mask.sum())
    twave_cols = ["t_wave_amplitude_mv", "rt_interval_ms", "qt_ms", "qtc_ms"]
    before_tw  = df.loc[twave_mask, ["animal_id", "status",
                                      "t_wave_excursion_mv"]].copy()
    df.loc[twave_mask, twave_cols] = float("nan")

    print(f"Rule B -- T-wave isolation FIXED (excursion < {T_WAVE_ISO_MIN_MV} mV): "f"{n_twave} animals -> {twave_cols} set to NaN")
    if n_twave:
        print(before_tw.to_string(index=False))

    # Confirm seed animals 201 and 146 survive Rule B
    nulled_ids = set(before_tw["animal_id"].tolist()) if n_twave else set()
    for a_id in [201, 146]:
        row = df[df["animal_id"] == a_id]
        if row.empty:
            print(f"  Animal {a_id}: not in dataset")
            continue
        r = row.iloc[0]
        flag = "NULLED" if a_id in nulled_ids else "OK -- NOT nulled"
        exc  = r.get("t_wave_excursion_mv", float("nan"))
        import math
        exc_str = f"{exc:.5f} mV" if not math.isnan(float(exc)) else "NaN"
        print(f"  Animal {a_id}: {flag}  t_wave_excursion_mv = {exc_str}")

    # --- Before/after NaN count per feature -----------------------------------
    print("=== Part B before/after: NaN count per feature ===" )
    watch_cols = ["rr_mean_ms", "qt_ms", "qtc_ms",
                  "t_wave_amplitude_mv", "rt_interval_ms"]
    nan_after  = df[watch_cols].isna().sum()
    print(nan_after.to_string())


In [ ]:
# =============================================================================
# DIAGNOSTIC: Rule B trigger -- post-fix verification (read-only)
# =============================================================================
# t_wave_amplitude_mv = max(signal in 40-80 ms window) -- a SIGNED peak vs zero.
# For inverted T-waves (common in mouse ECG), the window max is near baseline
# (~0 mV), so the trigger fires on normal animals.  This is wrong.
# Fix (Part B): use t_wave_excursion_mv = max|sig - baseline| (polarity-free).
# =============================================================================

if not df.empty:
    print("=== DIAGNOSTIC: Rule B trigger -- excursion-based, post-fix ===")
    print(f"Threshold: t_wave_amplitude_mv < {T_WAVE_ISO_MIN_MV} mV")
    print("(t_wave_excursion_mv = max|sig-baseline|; fires only on truly flat T-waves)")

    print(f"Animals nulled by current Rule B: {n_twave}")
    if n_twave:
        cols_show = ["animal_id", "status", "t_wave_amplitude_mv"]
        print(before_tw[cols_show].to_string(index=False))
    else:
        print("  (none)")

    nulled_ids = set(before_tw["animal_id"].tolist()) if n_twave else set()
    for a_id in [201, 146]:
        row = df[df["animal_id"] == a_id]
        if row.empty:
            print(f"Animal {a_id}: not in dataset")
            continue
        r = row.iloc[0]
        if a_id in nulled_ids:
            orig = before_tw.loc[before_tw["animal_id"] == a_id, "t_wave_amplitude_mv"].values
            orig_str = f"{orig[0]:.5f}" if len(orig) else "?"
            print(f"*** Animal {a_id}: NULLED by Rule B (original t_wave_amplitude_mv = {orig_str} mV) ***")
            print(f"    WRONG if {a_id} has a real T-wave -- confirms inverted-T bias.")
        else:
            tval = r.get("t_wave_amplitude_mv", float("nan"))
            import math
            tstr = f"{tval:.4f} mV" if not math.isnan(float(tval)) else "NaN (already null)"
            print(f"Animal {a_id}: NOT nulled -- t_wave_amplitude_mv = {tstr}  status={r['status']}")

    surv = df["t_wave_amplitude_mv"].dropna()
    print(f"t_wave_amplitude_mv distribution AFTER Rule B (n={len(surv)}):")
    for p in [5, 25, 50, 75, 95]:
        print(f"p{p:2d}: {float(surv.quantile(p/100)):.5f} mV")

    print("Diagnosis: if OK-status animals appear in the nulled list above,")
    print("Rule B fires on inverted T-waves, not truly flat T-waves.")
    print("Part B fix applied -- excursion trigger now in cell 8722be9c.")


### Change 3 — Missingness-indicator columns

For every feature that can be NaN, add a binary `<feature>_obs` column (1 = measured, 0 = null).
These become first-class features in the clustering matrix — the *pattern* of which features are unobservable is itself informative (hypothesis: one whole cluster may share a single missing feature).

In [ ]:
# Parts C-E: replace 13 collinear _obs columns with 4 cause-flags.
# Each flag maps to ONE distinct missingness mechanism:
#   beats_detected     -- 0 if FAILED_PEAKS (no peak extraction at all)
#   rhythm_regular     -- 0 if Rule A fired (rr_cv > RR_CV_IRREGULAR)
#   twave_isolated     -- 0 if Rule B fired (t_wave_excursion_mv < threshold)
#   enough_beats_for_sd -- 0 if per-beat SD is null despite beats existing
#
# Flags are kept 0/1 and NOT StandardScaled (Part D).  A documented weight
# constant FLAG_WEIGHT allows the balance to be tuned in one place.

# Weight applied to 0/1 flags before concatenation with standardised continuous.
# 1.0 means flags contribute on the same numeric scale as the unit-variance
# continuous columns; increase to emphasise missingness structure.
FLAG_WEIGHT = 1.0

if not df.empty:
    # Remove any stale _obs columns from previous design
    stale_obs = [c for c in df.columns if c.endswith("_obs")]
    if stale_obs:
        df.drop(columns=stale_obs, inplace=True)

    # beats_detected: 0 only for FAILED_PEAKS (no morphology extracted at all)
    df["beats_detected"]      = (df["status"] != "FAILED_PEAKS").astype(int)

    # rhythm_regular: 0 when rr_cv > RR_CV_IRREGULAR OR rr_cv is NaN (no beats)
    df["rhythm_regular"]      = (df["rr_cv"].notna() &
                                   (df["rr_cv"] <= RR_CV_IRREGULAR)).astype(int)

    # twave_isolated: 0 when Rule B fired (excursion below noise floor)
    df["twave_isolated"]      = (df["t_wave_excursion_mv"].notna() &
                                   (df["t_wave_excursion_mv"] >= T_WAVE_ISO_MIN_MV)).astype(int)

    # enough_beats_for_sd: 0 when qt_std_ms is NaN (too few individual QT estimates)
    df["enough_beats_for_sd"] = df["qt_std_ms"].notna().astype(int)

    CAUSE_FLAG_COLS = ["beats_detected", "rhythm_regular",
                       "twave_isolated", "enough_beats_for_sd"]

    print("=== Parts C-E: cause-flag summary (BEFORE -> AFTER) ===")
    print(f"Before: 13 collinear _obs columns (one per nullable feature)")
    print(f"After : 4 cause-flags (one per missingness mechanism)")
    print()
    for f in CAUSE_FLAG_COLS:
        n0 = int((df[f] == 0).sum())
        print(f"  {f:<22}: {n0:3d} animals flag=0 (mechanism active)")

    print()
    print(df[["animal_id", "status"] + CAUSE_FLAG_COLS].to_string(index=False))


## Part 4 — Clustering

## PCA — Dimensionality Check (before clustering)
Run this before interpreting cluster results.
If 2 PCs explain <70% of variance, cluster shapes in the 2D plot are misleading.
Check the scree plot to find the natural dimensionality of the data.

In [ ]:
# Build the clustering feature matrix (reused by the clustering cell below).
# Parts C-E design:
#   X_cont  -- 14 continuous features, median-imputed as a technical
#               necessity for StandardScaler/KMeans only.
#   X_flags -- 4 cause-flags (0/1 * FLAG_WEIGHT), NOT StandardScaled.
#               Each flag encodes one distinct missingness mechanism.
#   X_scaled = hstack(X_cont_scaled, X_flags)
# Three versions of X are also kept for the 3-way silhouette report.

if df.empty:
    print("No animals -- skipping PCA.")
else:
    cluster_feature_cols = [
        "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
        "r_amplitude_mv", "qrs_duration_ms",
        "j_wave_amplitude_mv", "t_wave_amplitude_mv",
        "qt_ms", "qtc_ms",
        "rt_interval_ms",
        "qt_std_ms",
        "qrs_std_ms",
        "r_amplitude_std_mv",
        "rr_cv",
    ]

    CAUSE_FLAG_COLS = ["beats_detected", "rhythm_regular",
                       "twave_isolated", "enough_beats_for_sd"]

    X_cont_raw    = df[cluster_feature_cols].copy()
    col_medians   = X_cont_raw.median(numeric_only=True)
    X_cont_filled = X_cont_raw.fillna(col_medians)
    X_cont_scaled = StandardScaler().fit_transform(X_cont_filled)

    # Cause-flags: 0/1, scaled only by FLAG_WEIGHT (not StandardScaler)
    X_flags = df[CAUSE_FLAG_COLS].values.astype(float) * FLAG_WEIGHT

    # Combined matrix
    X_scaled   = np.hstack([X_cont_scaled, X_flags])
    feat_names = cluster_feature_cols + CAUSE_FLAG_COLS
    n_features = X_scaled.shape[1]

    print(f"Clustering feature matrix: {X_scaled.shape[0]} animals x {n_features} features")
    print(f"  Continuous (standardised): {len(cluster_feature_cols)}")
    print(f"  Cause-flags (0/1 x {FLAG_WEIGHT}): {len(CAUSE_FLAG_COLS)}")
    n_null_cells = int(X_cont_raw.isna().sum().sum())
    print(f"  NaN cells in continuous block (imputed for numerics): {n_null_cells}")

    # ---- Scree plot ----
    max_pc = min(10, n_features)
    pca_full = PCA(n_components=max_pc, random_state=42).fit(X_scaled)
    cum_var  = np.cumsum(pca_full.explained_variance_ratio_)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(range(1, max_pc + 1), cum_var, marker="o", color="steelblue")
    ax.axhline(0.80, color="orange", linestyle="--", linewidth=1, label="80% variance")
    ax.axhline(0.90, color="red",    linestyle="--", linewidth=1, label="90% variance")
    ax.set_xlabel("Number of principal components")
    ax.set_ylabel("Cumulative variance explained")
    ax.set_title("PCA Scree (continuous + 4 cause-flags)")
    ax.set_xticks(range(1, max_pc + 1)); ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.3); ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "11_pca_scree.png", dpi=150)
    plt.show()

    print("Variance explained:")
    for k in (2, 3, 5):
        if k <= max_pc:
            print(f"  {k} PCs -> {cum_var[k-1]*100:5.1f}%")

    # ---- 2D PCA scatter coloured by k=5 k-means ----
    pca2    = PCA(n_components=2, random_state=42)
    scores2 = pca2.fit_transform(X_scaled)
    pc1_pct, pc2_pct = pca2.explained_variance_ratio_[:2] * 100
    _preview_k5 = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_scaled)

    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(scores2[:, 0], scores2[:, 1], c=_preview_k5,
                    cmap="tab10", s=40, edgecolor="k", linewidth=0.3)
    nr_mask = df["status"] == "NEEDS_REVIEW"
    if nr_mask.any():
        ax.scatter(scores2[nr_mask.values, 0], scores2[nr_mask.values, 1],
                   marker="^", s=80, color="black", zorder=5, label="NEEDS_REVIEW")
    ax.set_xlabel(f"PC1 ({pc1_pct:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pc2_pct:.1f}% var)")
    ax.set_title("PCA 2D -- continuous + 4 cause-flags, coloured by k=5")
    ax.grid(alpha=0.3)
    ax.add_artist(ax.legend(*sc.legend_elements(), title="cluster_k5", loc="best", fontsize=8))
    if nr_mask.any():
        ax.legend(loc="upper right", fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "12_pca_scatter_2d.png", dpi=150)
    plt.show()


In [ ]:
# --- Labelled PCA figure (outliers for review) -------------------------------
# Re-runs 2-component PCA on the SAME X_scaled matrix built above; every point is
# labelled with its animal ID and statistical outliers are highlighted for review.
if df.empty:
    print("No animals — skipping PCA figure.")
else:
    _pca_r = PCA(n_components=2, random_state=42)
    _scores = _pca_r.fit_transform(X_scaled)
    _pc1_pct, _pc2_pct = _pca_r.explained_variance_ratio_[:2] * 100

    # Colour by k=5 k-means: use the saved column if present, otherwise the same
    # seed (identical labels to cluster_k5 computed in the clustering cell).
    if "cluster_k5" in df.columns:
        _labels_k5 = df["cluster_k5"].to_numpy()
    else:
        _labels_k5 = KMeans(n_clusters=5, n_init=10, random_state=42).fit_predict(X_scaled)
    _vmin, _vmax = int(_labels_k5.min()), int(_labels_k5.max())

    _animal_ids = df["animal_id"].to_numpy()

    # Outlier rule (pulled from data, not hardcoded): PC1 > 5 OR |PC2| > 3
    _is_outlier = (_scores[:, 0] > 5) | (np.abs(_scores[:, 1]) > 3)
    _outlier_ids = [int(a) for a in _animal_ids[_is_outlier]]

    fig, ax = plt.subplots(figsize=(14, 10))

    # Normal points (default size) then outliers (size 120, black edge); shared colour scale
    _norm = ~_is_outlier
    ax.scatter(_scores[_norm, 0], _scores[_norm, 1], c=_labels_k5[_norm],
               cmap="tab10", vmin=_vmin, vmax=_vmax, s=40, alpha=0.85)
    ax.scatter(_scores[_is_outlier, 0], _scores[_is_outlier, 1], c=_labels_k5[_is_outlier],
               cmap="tab10", vmin=_vmin, vmax=_vmax, s=120,
               edgecolor="black", linewidth=1.5, zorder=5)

    # Label every point with its animal ID, offset so labels don't sit on the dot
    for _x, _y, _aid in zip(_scores[:, 0], _scores[:, 1], _animal_ids):
        ax.text(_x + 0.15, _y + 0.1, str(int(_aid)), fontsize=7, alpha=0.9)

    ax.set_xlabel(f"PC1 ({_pc1_pct:.1f}% variance)", fontsize=12)
    ax.set_ylabel(f"PC2 ({_pc2_pct:.1f}% variance)", fontsize=12)
    ax.grid(alpha=0.3)

    # Title + smaller subtitle
    fig.suptitle(f"PCA of baseline ECG features — {len(df)} animals",
                 fontsize=15, fontweight="bold")
    ax.set_title("Point labels = animal ID. Highlighted points = statistical outliers in "
                 "PCA space. To be interpreted against treatment metadata.",
                 fontsize=9)

    # Outlier text box, bottom-right corner
    _box = "Outliers to discuss: " + (", ".join(str(a) for a in _outlier_ids)
                                      if _outlier_ids else "none")
    ax.text(0.98, 0.02, _box, transform=ax.transAxes, fontsize=10, ha="right", va="bottom",
            bbox=dict(boxstyle="round", facecolor="wheat", edgecolor="black", alpha=0.9))

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(FIGURES_DIR / "pca_outliers.png", dpi=150)
    plt.show()

    print(f"PCA outliers (PC1 > 5 or |PC2| > 3): {_outlier_ids}")


In [ ]:
if df.empty:
    print("No animals to cluster.")
else:
    # X_scaled, X_cont_scaled, X_flags, feat_names built in the PCA cell above.
    CAUSE_FLAG_COLS = ["beats_detected", "rhythm_regular",
                       "twave_isolated", "enough_beats_for_sd"]

    # ---- 1) k-means: k=2 and k=5 on combined matrix ----
    def run_kmeans(k, X=X_scaled):
        if len(df) < k:
            return np.full(len(df), -1)
        return KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)

    df["cluster_k2"] = run_kmeans(2)
    df["cluster_k5"] = run_kmeans(5)
    print("k-means  k=2 sizes:", dict(df["cluster_k2"].value_counts()))
    print("k-means  k=5 sizes:", dict(df["cluster_k5"].value_counts()))
    if "study_guess" in df.columns:
        print("\ncluster_k2 vs study_guess:")
        print(pd.crosstab(df["cluster_k2"], df["study_guess"]))
    if "status" in df.columns:
        print("\ncluster_k5 vs status:")
        print(pd.crosstab(df["cluster_k5"], df["status"]))

    # ---- 2) Three-way silhouette sweep (Part E) ----
    # continuous-only, flags-only, combined -- so contribution of missingness
    # structure is visible rather than assumed.
    sil_ks = list(range(2, 9))
    sil_results = {"continuous": [], "flags": [], "combined": []}

    for k in sil_ks:
        for label, X_in in [("continuous", X_cont_scaled),
                            ("flags",      X_flags),
                            ("combined",   X_scaled)]:
            if len(df) < k:
                sil_results[label].append(float("nan")); continue
            labs = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_in)
            sil_results[label].append(silhouette_score(X_in, labs))

    print("\n=== Part E: silhouette three ways (k=2..8) ===")
    hdr = f"{'k':>2}  {'continuous':>12}  {'flags':>10}  {'combined':>10}"
    print(hdr)
    for i, k in enumerate(sil_ks):
        sc_c = sil_results["continuous"][i]
        sc_f = sil_results["flags"][i]
        sc_b = sil_results["combined"][i]
        print(f"{k:2d}  {sc_c:12.3f}  {sc_f:10.3f}  {sc_b:10.3f}")

    best_k_comb = sil_ks[int(np.argmax(sil_results['combined']))]
    print(f"\n-> best k (combined): {best_k_comb}  "
          f"(sil {max(sil_results['combined']):.3f})")

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, col, ls in [("continuous", "steelblue", "-"),
                            ("flags",      "orange",    "--"),
                            ("combined",   "green",     "-.")]:
        ax.plot(sil_ks, sil_results[label], marker="o", color=col,
                linestyle=ls, label=label)
    ax.axvline(best_k_comb, color="green", linestyle=":", linewidth=1)
    ax.set_xlabel("k"); ax.set_ylabel("Mean silhouette score")
    ax.set_title("Silhouette: continuous / flags / combined")
    ax.grid(alpha=0.3); ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "13_silhouette.png", dpi=150)
    plt.show()

    # ---- 3) Does any cause-flag predict cluster membership? ----
    print("\n=== Part E: cause-flag rates per cluster (k=5) ===")
    flag_rates = df.groupby("cluster_k5")[CAUSE_FLAG_COLS].mean()
    print(flag_rates.round(2).to_string())
    print("\nHypothesis: a cluster with rate ~0 on one flag is defined")
    print("by that missing mechanism (not random noise).")
    for col in CAUSE_FLAG_COLS:
        for cl, rate in flag_rates[col].items():
            if rate < 0.20:
                n_cl = int((df["cluster_k5"] == cl).sum())
                print(f"  PROVISIONAL: cluster {cl} (n={n_cl}) has {col}=0 "
                      f"in {100*(1-rate):.0f}% of animals")

    # ---- 4) Hierarchical + GMM ----
    Z = linkage(X_scaled, method="ward")
    df["cluster_hier_k2"] = fcluster(Z, t=2, criterion="maxclust")
    df["cluster_hier_k5"] = fcluster(Z, t=5, criterion="maxclust")
    df["cluster_gmm_k2"]  = GaussianMixture(n_components=2, random_state=42).fit_predict(X_scaled)
    df["cluster_gmm_k5"]  = GaussianMixture(n_components=5, random_state=42).fit_predict(X_scaled)
    print("\nHierarchical k=5:", dict(df["cluster_hier_k5"].value_counts()))
    print("GMM k=5:          ", dict(df["cluster_gmm_k5"].value_counts()))

### Change 5 — Missingness matrix (animals × features, sorted by cluster)

The key visual: **black = feature is null, white = feature was measured**.
If missingness is structured (not random), you will see horizontal bands of black
aligned with cluster boundaries — one cluster may be entirely defined by a shared
missing feature.

In [ ]:
if df.empty or "cluster_k5" not in df.columns:
    print("Run clustering cell first.")
else:
    # Missingness matrix: continuous features (null = black) + 4 cause-flags,
    # sorted by k=5 cluster.  Red lines = cluster boundaries.
    CAUSE_FLAG_COLS = ["beats_detected", "rhythm_regular",
                       "twave_isolated", "enough_beats_for_sd"]
    cont_cols = [
        "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
        "r_amplitude_mv", "qrs_duration_ms",
        "j_wave_amplitude_mv", "t_wave_amplitude_mv",
        "qt_ms", "qtc_ms", "rt_interval_ms",
        "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv", "rr_cv",
    ]
    cont_cols  = [c for c in cont_cols  if c in df.columns]
    flag_cols  = [c for c in CAUSE_FLAG_COLS if c in df.columns]
    all_cols   = cont_cols + flag_cols

    df_sorted = df.sort_values(["cluster_k5", "animal_id"]).reset_index(drop=True)

    cont_mat = df_sorted[cont_cols].notna().astype(int).values
    flag_mat = df_sorted[flag_cols].values.astype(int)
    miss_mat = np.hstack([cont_mat, flag_mat])

    cluster_seq = df_sorted["cluster_k5"].values
    boundaries  = [i for i in range(1, len(cluster_seq))
                   if cluster_seq[i] != cluster_seq[i - 1]]

    n_cont = len(cont_cols); n_flag = len(flag_cols)
    fig, ax = plt.subplots(figsize=(max(12, 0.7 * len(all_cols)),
                                    max(8, 0.18 * len(df_sorted))))
    im = ax.imshow(miss_mat, aspect="auto", cmap="binary_r",
                   vmin=0, vmax=1, interpolation="nearest")
    ax.set_xticks(np.arange(-0.5, len(all_cols), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(df_sorted), 1), minor=True)
    ax.grid(which="minor", color="lightgray", linewidth=0.3)
    for b in boundaries:
        ax.axhline(b - 0.5, color="red", linewidth=1.5)
    ax.axvline(n_cont - 0.5, color="royalblue", linewidth=2, linestyle="--")
    ax.text(n_cont - 0.5, -0.8, "flags ->", color="royalblue", fontsize=8, ha="center")

    short = lambda s: (s.replace("_amplitude_mv","_amp").replace("_duration_ms","_dur")
                        .replace("_interval_ms","_int").replace("_ms","")
                        .replace("_mv","").replace("heart_rate_bpm","HR_bpm"))
    ax.set_xticks(range(len(all_cols)))
    ax.set_xticklabels([short(c) for c in all_cols], rotation=45, ha="right", fontsize=8)
    ytick_labels = [f"{int(r.animal_id)}  [k{int(r.cluster_k5)}]"
                    for _, r in df_sorted.iterrows()]
    ax.set_yticks(range(len(df_sorted)))
    ax.set_yticklabels(ytick_labels, fontsize=7)

    prev_c = None; c_start = 0
    for i, c in enumerate(cluster_seq):
        if c != prev_c:
            if prev_c is not None:
                ax.text(len(all_cols) + 0.3, (c_start + i - 1) / 2,
                        f"cl {int(prev_c)}", va="center", fontsize=8, color="red")
            c_start = i; prev_c = c
    ax.text(len(all_cols) + 0.3, (c_start + len(cluster_seq) - 1) / 2,
            f"cl {int(prev_c)}", va="center", fontsize=8, color="red")

    ax.set_xlabel("Feature (left of blue line) | Cause-flag (right)", fontsize=10)
    ax.set_ylabel("Animal (sorted by k=5 cluster)", fontsize=10)
    ax.set_title("Missingness matrix -- black=null (continuous) or flag=0 (mechanism active)\n"
                 "Red lines = cluster boundaries.  Blue line = flag block start.", fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.3, label="0=null/flagged  1=observed/ok")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "14_missingness_matrix.png", dpi=150, bbox_inches="tight")
    plt.show()

    print("\n=== Missingness rate per feature per cluster (k=5) ===")
    null_rate = df.groupby("cluster_k5")[cont_cols].apply(lambda g: g.isna().mean())
    flag_rate = df.groupby("cluster_k5")[flag_cols].mean()
    with pd.option_context("display.float_format", "{:.2f}".format, "display.width", 200):
        print("Continuous null rates:")
        print(null_rate.T.to_string())
        print("Cause-flag rates (1=mechanism absent, 0=active):")
        print(flag_rate.T.to_string())

    print("\n=== PROVISIONAL: clusters where a cause-flag is ~constant ===")
    found = False
    for feat in flag_cols:
        for cl, rate in flag_rate[feat].items():
            if rate < 0.20:
                n_cl = int((df["cluster_k5"] == cl).sum())
                print(f"  cluster {cl} (n={n_cl}): {feat}=0 in "
                      f"{100*(1-rate):.0f}% of animals")
                print(f"    -> PROVISIONAL: this cluster may be defined by {feat}=0")
                print(f"       (technical vs biological unresolved -- pending clinical review)")
                found = True
    if not found:
        print("  No cluster has a cause-flag uniformly 0.")
        print("  Missingness does not uniquely define any cluster at this threshold.")

In [ ]:
if not df.empty and templates:
    clusters_present = sorted(c for c in df["cluster_k5"].unique() if c >= 0)
    n_panels = max(len(clusters_present), 1)
    fig, axes = plt.subplots(1, n_panels, figsize=(3.5 * n_panels, 4),
                             sharey=True, squeeze=False)
    axes = axes[0]
    last_t_ms = None
    for ax, c in zip(axes, clusters_present):
        members = df.loc[df["cluster_k5"] == c, "animal_id"].tolist()
        stack = []
        for aid in members:
            if aid in templates:
                tmpl, t_ms = templates[aid]
                ax.plot(t_ms, tmpl, color="gray", alpha=0.4, linewidth=0.7)
                stack.append(tmpl); last_t_ms = t_ms
        if stack and last_t_ms is not None:
            centroid = np.mean(np.vstack(stack), axis=0)
            ax.plot(last_t_ms, centroid, color="red", linewidth=2.0, label="cluster mean")
        ax.set_title(f"k=5 cluster {c}  (n={len(members)})", fontsize=9)
        ax.set_xlabel("Time from R peak (ms)")
        ax.axvline(0, color="k", linewidth=0.5, linestyle="--", alpha=0.5)
        ax.grid(alpha=0.3); ax.legend(fontsize=7)
    axes[0].set_ylabel("Voltage (mV)")
    fig.suptitle("Averaged-beat templates by k=5 cluster", y=1.02)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "10_cluster_beats.png", dpi=150, bbox_inches="tight")
    plt.show()

## Part 5 — Master CSV

In [ ]:
if not df.empty:
    out_cols = [
        "animal_id", "recording_date", "study_guess", "status",
        "n_beats", "heart_rate_bpm", "rr_mean_ms", "rr_std_ms",
        "median_rr_ms", "pct_near_floor",
        "r_amplitude_mv", "qrs_duration_ms",
        "j_wave_amplitude_mv", "t_wave_amplitude_mv",
        "qt_ms", "qtc_ms", "qtc_formula",
        "qt_std_ms", "qrs_std_ms", "r_amplitude_std_mv", "rr_cv",
        "cluster_k2", "cluster_k5",
        "cluster_hier_k2", "cluster_hier_k5",
        "cluster_gmm_k2", "cluster_gmm_k5",
    ]
    summary = df[[c for c in out_cols if c in df.columns]].copy()
    summary["recording_date"] = pd.to_datetime(summary["recording_date"]).dt.date
    csv_path = OUTPUTS_DIR / "all_animals_features.csv"
    summary.to_csv(csv_path, index=False)
    print(f"Saved: {csv_path}")
    print(f"Rows : {len(summary)}")
    summary.head(20)
else:
    print("No animals to save.")


## Final summary

In [ ]:
print("=" * 60)
print("NOTEBOOK 02 — SUMMARY")
print("=" * 60)
status_counts = pd.Series([r["status"] for r in rows_all]).value_counts()
print(f"Total files found      : {len(files)}")
for status, n in status_counts.items():
    print(f"  {status:<20} : {n}")

if per_animal:
    hrs = pd.Series([a["heart_rate_bpm"] for a in per_animal])
    print(f"\nHR (bpm)        : min {hrs.min():.0f}   mean {hrs.mean():.0f}   max {hrs.max():.0f}")
    print(f"In 400-500 bpm  : {((hrs >= HR_NORMAL_LOW) & (hrs <= HR_NORMAL_HIGH)).sum()}/{len(hrs)}")

if needs_review:
    print("\nNeeds manual review (HR outside 300-700 bpm):")
    for a in needs_review:
        print(f"   - animal {a['animal_id']:>4}  HR {a['heart_rate_bpm']:6.1f} bpm  ({a['source_file']})")

failed = [r for r in rows_all if r["status"].startswith("FAILED")]
if failed:
    print("\nFailed files (flagged for review):")
    for r in failed[:40]:
        print(f"   - {r['file']}: {r['status']} — {r['rule_used']}")
    if len(failed) > 40:
        print(f"   ... and {len(failed) - 40} more (see outputs/per_file_audit.csv)")

print("\nFigures saved to:", FIGURES_DIR)
print("Feature CSV     :", OUTPUTS_DIR / "all_animals_features.csv")
print("Per-file audit  :", OUTPUTS_DIR / "per_file_audit.csv")

## 3D PCA and nonlinear (t-SNE) embedding

The 2D PCA is a linear projection and could hide structure. Here we view the
first three principal components in 3D and a nonlinear **t-SNE** embedding of the
same `X_scaled` matrix. If treatment groups existed they would appear in at least
one of these views; only a data-quality gradient does - confirming that the
absence of grouping is a property of the data, not an artefact of PCA's linearity.


In [ ]:
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# clean vs flagged, from the cause-flags already in df
_clean = ((df["status"] == "OK") & (df["rhythm_regular"] == 1)
          & (df["twave_isolated"] == 1) & (df["enough_beats_for_sd"] == 1)).to_numpy()
_col = np.where(_clean, "#2a9d5c", "#c9432b")

_p3 = PCA(n_components=3, random_state=42).fit(X_scaled)
_P3 = _p3.transform(X_scaled)
_evr = _p3.explained_variance_ratio_
_perp = max(5, min(30, (len(df) - 1) // 3))
_TS = TSNE(n_components=2, perplexity=_perp, init="pca",
           learning_rate="auto", random_state=42).fit_transform(X_scaled)

_fig = plt.figure(figsize=(15, 6.5))
_ax1 = _fig.add_subplot(1, 2, 1, projection="3d")
for _lab, _m, _c in [(f"clean (n={int(_clean.sum())})", _clean, "#2a9d5c"),
                     (f"flagged (n={int((~_clean).sum())})", ~_clean, "#c9432b")]:
    _ax1.scatter(_P3[_m, 0], _P3[_m, 1], _P3[_m, 2], s=30, c=_c, label=_lab,
                 edgecolor="k", linewidth=0.3)
_ax1.set_xlabel(f"PC1 ({_evr[0]:.0%})"); _ax1.set_ylabel(f"PC2 ({_evr[1]:.0%})")
_ax1.set_zlabel(f"PC3 ({_evr[2]:.0%})")
_ax1.set_title(f"3D PCA - first 3 PCs ({_evr[:3].sum():.0%} variance)\nquality gradient, no treatment groups")
_ax1.legend(loc="upper left")

_ax2 = _fig.add_subplot(1, 2, 2)
_ax2.scatter(_TS[:, 0], _TS[:, 1], s=32, c=_col, edgecolor="k", linewidth=0.3)
_ax2.set_xlabel("t-SNE 1"); _ax2.set_ylabel("t-SNE 2")
_ax2.set_title(f"Nonlinear reduction (t-SNE, perplexity={_perp})\nno discrete clusters emerge either")
_ax2.grid(alpha=0.2)
from matplotlib.lines import Line2D
_ax2.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc="#2a9d5c", mec="k", label="clean"),
                     Line2D([0], [0], marker="o", ls="", mfc="#c9432b", mec="k", label="flagged")])
_fig.suptitle("Linear (3D PCA) and nonlinear (t-SNE) views agree: only a data-quality axis",
              fontweight="bold")
_fig.tight_layout()
_fig.savefig(FIGURES_DIR / "14_pca3d_tsne.png", dpi=150)
plt.show()
print(f"PC1-3 explain {_evr[:3].sum():.1%} of variance; t-SNE perplexity {_perp}")


## Problem-animal figures and triage

Regenerates one raw-vs-filtered figure per flagged recording and builds the triage
table. **All statistics are computed over the full baseline window**; the plots show
a 1-second slice for legibility only.

The triage is keyed on `rr_cv`, `status` and heart rate -- the pipeline's own
full-window metrics. (Raw-vs-filtered correlation and "% removed" are reported for
information but are *not* used to triage: on full windows they separate clean from
flagged only weakly, with heavy overlap.)


In [ ]:
_PA = FIGURES_DIR / "problem_animals"; _PA.mkdir(parents=True, exist_ok=True)
_rms = lambda x: float(np.sqrt(np.mean(np.square(x))))

_clean_mask = ((df["status"] == "OK") & (df["rhythm_regular"] == 1)
               & (df["twave_isolated"] == 1) & (df["enough_beats_for_sd"] == 1))
_flagged = df[~_clean_mask]

def _tier(status, hr, rr_cv):
    if status != "OK" or not (HR_ACCEPT_LOW <= hr <= HR_ACCEPT_HIGH) or rr_cv > 0.60:
        return "DROP"
    return "KEEP" if rr_cv <= 0.25 else "REVIEW"

_rows = []
for _, _a in _flagged.iterrows():
    _p = DATA_DIR / str(_a["source_file"])
    if not _p.exists():
        continue
    _nh, _lead, _col, _ = parse_header_and_layout(_p)
    _v, _mk = load_ecg_file(_p, _nh, _col)
    _s, _e = find_baseline_window(_mk, len(_v))[:2]
    _raw = _v[_s:_e]
    _filt = bandpass_filter(_raw)                 # filter-only (no polarity flip)
    _pk, _inv, _hr, _ = detect_r_peaks(_filt)
    # --- full-window statistics ---
    _r = _raw - np.mean(_raw); _f = _filt - np.mean(_filt)
    _corr = float(np.corrcoef(_r, _f)[0, 1]) if _rms(_r) else np.nan
    _pct = 100.0 * _rms(_r - _f) / _rms(_r) if _rms(_r) else np.nan
    _rr = np.diff(_pk) * (1000.0 / FS)
    _cv = float(np.std(_rr) / np.mean(_rr)) if len(_pk) > 2 else np.nan
    _t = _tier(_a["status"], float(_hr), _cv if _cv == _cv else 99)
    _rows.append(dict(animal=int(_a["animal_id"]), tier=_t, status=_a["status"],
                      hr=round(float(_hr)), rr_cv=round(_cv, 3), beats=len(_pk),
                      window_s=round(len(_raw) / FS, 1),
                      corr_full=round(_corr, 3), pct_removed_full=round(_pct),
                      inverted=bool(_inv)))
    # --- figure: 1 s slice plotted, but title statistics are FULL-window ---
    _disp = -_filt if _inv else _filt
    _n = min(int(1.0 * FS), len(_raw)); _ts = np.arange(_n) / FS
    _fig, _ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    _ax[0].plot(_ts, _r[:_n], color="#999", lw=0.7, label="raw (de-meaned)")
    _ax[0].plot(_ts, _disp[:_n], color="#c0392b", lw=0.8, label="filtered")
    _pin = _pk[_pk < _n]
    _ax[0].plot(_pin / FS, _disp[_pin], "v", color="#16a085", ms=6, label="detected R")
    _ax[0].set_title(f"Animal {int(_a['animal_id'])} [{_t}] - HR {_hr:.0f} bpm, "
                     f"rr_cv {_cv:.3f}{' , INVERTED' if _inv else ''}   "
                     f"(full {len(_raw)/FS:.0f}s window; 1 s shown)", fontweight="bold")
    _ax[0].set_ylabel("mV"); _ax[0].legend(loc="upper right", fontsize=7)
    _ax[1].plot(_ts, (_r - _f)[:_n], color="#2c3e50", lw=0.6)
    _ax[1].set_title(f"Filter residual - {_pct:.0f}% of the full window removed "
                     f"(corr {_corr:+.2f}; informational, not a triage criterion)",
                     fontweight="bold")
    _ax[1].set_ylabel("mV"); _ax[1].set_xlabel("time (s)")
    for _a2 in _ax: _a2.grid(alpha=0.2)
    _fig.tight_layout()
    _fig.savefig(_PA / f"problem_A{int(_a['animal_id'])}.png", dpi=120)
    plt.close(_fig)

triage = pd.DataFrame(_rows).sort_values(["tier", "rr_cv"])
triage.to_csv(OUTPUTS_DIR / "flagged_triage.csv", index=False)
print(triage["tier"].value_counts().to_string())
print(f"\n{len(triage)} figures -> {_PA}")
for _t in ["KEEP", "REVIEW", "DROP"]:
    _g = triage[triage.tier == _t]
    print(f"\n{_t} (n={len(_g)}): {sorted(_g.animal.tolist())}")


## Pipeline validation gallery - the 10 cleanest recordings

Evidence that preprocessing and feature extraction work. The ten animals below are
the highest-scoring clean recordings (regular rhythm, physiological heart rate, QTc
near the mouse literature value, consistent beat-to-beat QT).

For each: the **raw** signal and the **filtered** signal overlaid with every
**detected R-peak**, then the **averaged beat** with the extracted landmarks
(P, Q, R, J, S, T) and intervals marked. The values printed on each panel come from
`extract_morphology_features` -- the same function that builds the feature table, so
what is drawn is exactly what is measured.


In [ ]:
BEST10 = [211, 206, 239, 247, 208, 218, 248, 142, 230, 136]

_GAL = FIGURES_DIR / "validation_gallery"; _GAL.mkdir(parents=True, exist_ok=True)
# Landmark windows verified against the averaged-beat morphology of all ten
# animals. True mouse sequence: P -> Q -> R -> S(+5) -> J(+8..10) -> T(+20,
# inverted) -> repolarisation complete (~+31..40). NOTE: these differ from the
# windows used by _measure_one, which place S at +20 (that is really the T wave)
# and search for T at 40-80 ms (which at mouse rates lands on the NEXT beat's
# P wave). Pending confirmation of the correct convention, the figure marks the
# waves where they physically are.
_LM = [("P", -70, -35, "max", "#8e44ad"), ("Q", -14, -2, "min", "#2980b9"),
       ("R",   0,   0, "r",   "#c9432b"), ("S",   0,   8, "min", "#16a085"),
       ("J",   4,  16, "max", "#d35400"), ("T",  14,  30, "min", "#2c3e50")]

def _landmark(tmpl, t_ms, lo, hi, mode):
    if mode == "r":
        return 0.0
    m = (t_ms >= lo) & (t_ms <= hi)
    if not m.any():
        return None
    tt, seg = t_ms[m], tmpl[m]
    return float(tt[np.argmax(seg) if mode == "max" else np.argmin(seg)])

def _prep(aid):
    """Run the real pipeline on one animal; return everything needed to plot."""
    _row = df[df.animal_id == aid]
    _p = DATA_DIR / str(_row.iloc[0]["source_file"])
    _nh, _l, _c, _ = parse_header_and_layout(_p)
    _v, _mk = load_ecg_file(_p, _nh, _c)
    _s, _e = find_baseline_window(_mk, len(_v))[:2]
    _filt = bandpass_filter(_v[_s:_e])
    _pk, _inv, _hr, _ = detect_r_peaks(_filt)
    if _inv:
        _filt = -_filt
    _rr = np.diff(_pk) * (1000.0 / FS)
    _tmpl, _t_ms, _beats = average_beats(_filt, _pk)
    _f = extract_morphology_features(_tmpl, _t_ms, _beats, rr_ms=float(np.median(_rr)))
    _f["heart_rate_bpm"] = float(_hr)
    _f["qtc_ms"] = _f["qt_ms"] / np.sqrt(float(np.mean(_rr)) / 100.0)
    return dict(raw=_v[_s:_e] - np.mean(_v[_s:_e]), filt=_filt, pk=_pk,
                tmpl=_tmpl, t_ms=_t_ms, feats=_f, n=len(_pk))

_summary = []
for _aid in BEST10:
    _d = _prep(_aid); _f = _d["feats"]; _tmpl, _t_ms = _d["tmpl"], _d["t_ms"]
    _base = float(np.median(_tmpl[_t_ms < -70])) if (_t_ms < -70).any() else 0.0
    _fig, _ax = plt.subplots(2, 1, figsize=(12, 8))

    # --- panel 1: raw vs filtered with detected beats ---
    _nn = min(int(1.5 * FS), len(_d["filt"])); _ts = np.arange(_nn) / FS
    _ax[0].plot(_ts, _d["raw"][:_nn], color="#bbb", lw=0.7, label="raw")
    _ax[0].plot(_ts, _d["filt"][:_nn], color="#c0392b", lw=0.9, label="filtered")
    _pin = _d["pk"][_d["pk"] < _nn]
    _ax[0].plot(_pin / FS, _d["filt"][_pin], "v", color="#16a085", ms=8,
                label=f"{_d['n']} R-peaks detected")
    _ax[0].set_title(f"Animal {_aid} - every beat detected  "
                     f"(HR {_f['heart_rate_bpm']:.0f} bpm)", fontweight="bold")
    _ax[0].set_ylabel("mV"); _ax[0].set_xlabel("time (s)")
    _ax[0].legend(loc="upper right", fontsize=8); _ax[0].grid(alpha=.25)

    # --- panel 2: averaged beat with extracted landmarks ---
    _ax[1].plot(_t_ms, _tmpl, color="black", lw=1.8)
    _ax[1].axhline(_base, color="gray", ls=":", lw=1, alpha=.7)
    _pts = {}
    for _nm, _lo, _hi, _md, _col in _LM:
        _tm = _landmark(_tmpl, _t_ms, _lo, _hi, _md)
        if _tm is None:
            continue
        _pts[_nm] = _tm
        _i = int(np.argmin(np.abs(_t_ms - _tm)))
        _ax[1].plot(_tm, _tmpl[_i], "o", color=_col, ms=9, zorder=5)
        _ax[1].annotate(_nm, (_tm, _tmpl[_i]), textcoords="offset points",
                        xytext=(0, 10), ha="center", fontweight="bold", color=_col)
    if "Q" in _pts and not np.isnan(_f["qt_ms"]):        # QT interval bracket
        _y = _tmpl.min() - 0.05 * (_tmpl.max() - _tmpl.min())
        _ax[1].annotate("", xy=(_pts["Q"] + _f["qt_ms"], _y), xytext=(_pts["Q"], _y),
                        arrowprops=dict(arrowstyle="<->", color="#c0392b", lw=1.8))
        _ax[1].text(_pts["Q"] + _f["qt_ms"] / 2, _y, f" QT {_f['qt_ms']:.0f} ms",
                    ha="center", va="bottom", color="#c0392b", fontweight="bold")
    _ax[1].set_xlim(-100, 110); _ax[1].set_xlabel("time relative to R-peak (ms)")
    _ax[1].set_ylabel("mV"); _ax[1].grid(alpha=.25)
    _ax[1].set_title("Averaged beat with extracted features", fontweight="bold")
    _tamp = np.nan
    if "T" in _pts:
        _ti = int(np.argmin(np.abs(_t_ms - _pts["T"])))
        _tamp = float(_tmpl[_ti] - _base)          # inverted T, measured at the marker
    _txt = (f"HR {_f['heart_rate_bpm']:.0f} bpm | QT {_f['qt_ms']:.1f} ms | "
            f"QTc {_f['qtc_ms']:.1f} ms | R {_f['r_amplitude_mv']:.2f} mV | "
            f"T {_tamp:+.3f} mV (inverted)")
    _ax[1].text(0.5, -0.28, _txt, transform=_ax[1].transAxes, ha="center", fontsize=9,
                bbox=dict(boxstyle="round", fc="#f4f4f4", ec="#bbb"))
    _fig.tight_layout(rect=[0, 0.02, 1, 1])
    _fig.savefig(_GAL / f"validation_A{_aid}.png", dpi=140)
    plt.show(); plt.close(_fig)
    _summary.append(dict(animal=_aid, hr=round(_f["heart_rate_bpm"]),
                         qrs_ms=_f["qrs_duration_ms"], qt_ms=round(_f["qt_ms"], 1),
                         qtc_ms=round(_f["qtc_ms"], 1),
                         r_mv=round(_f["r_amplitude_mv"], 2), beats=_d["n"]))

# --- consistency summary across the ten ---
gallery = pd.DataFrame(_summary)
print(gallery.to_string(index=False))
print(f"\nQTc across the 10: {gallery.qtc_ms.mean():.1f} +/- {gallery.qtc_ms.std():.1f} ms "
      f"(mouse literature ~41 ms)")
print(f"QT  across the 10: {gallery.qt_ms.mean():.1f} +/- {gallery.qt_ms.std():.1f} ms")
gallery.to_csv(OUTPUTS_DIR / "validation_gallery.csv", index=False)
print(f"\n{len(BEST10)} figures -> {_GAL}")


## Beat-level structure: full recording, single beat, and all beats overlaid

The mouse-ECG analogue of the standard functional-data presentation used for
kinematic gait data (Tackney et al., 2026, Fig. 6), where a full recording, one
representative cycle, and all individual cycles overlaid per subject are shown
together.

- **(A)** a full filtered recording with every detected beat marked;
- **(B)** the averaged beat with the extracted landmarks;
- **(C)** *every individual beat* overlaid for each animal, with the average on
  top. The top row shows clean animals, the bottom row flagged ones.

Panel (C) is the informative one: a **tight bundle** means the beat morphology is
consistent and the averaged template is a fair summary; a **wide, scattered
bundle** means beats are not aligning, which is exactly what the rhythm gate
detects. It shows within-animal consistency and between-animal variability
directly, rather than through a summary statistic.


In [ ]:
FIG6_A   = 211                                  # panel A/B exemplar (clean)
FIG6_CLEAN   = [211, 206, 239, 247]             # panel C, top row
FIG6_FLAGGED = [111, 113, 120, 107]             # panel C, bottom row

def _fig6_prep(aid):
    _row = df[df.animal_id == aid]
    if _row.empty:
        return None
    _p = DATA_DIR / str(_row.iloc[0]["source_file"])
    _nh, _l, _c, _ = parse_header_and_layout(_p)
    _v, _mk = load_ecg_file(_p, _nh, _c)
    _s, _e = find_baseline_window(_mk, len(_v))[:2]
    _filt = bandpass_filter(_v[_s:_e])
    _pk, _inv, _hr, _ = detect_r_peaks(_filt)
    if _inv:
        _filt = -_filt
    if len(_pk) < 3:
        return None
    _rr = np.diff(_pk) * (1000.0 / FS)
    _tmpl, _t_ms, _beats = average_beats(_filt, _pk)
    _base = float(np.median(_tmpl[_t_ms < -70])) if (_t_ms < -70).any() else 0.0
    return dict(filt=_filt, pk=_pk, tmpl=_tmpl - _base, t_ms=_t_ms,
                beats=(None if _beats is None else _beats - _base),
                hr=float(_hr), rr=float(np.median(_rr)),
                cv=float(np.std(_rr) / np.mean(_rr)))

_d = _fig6_prep(FIG6_A)
fig = plt.figure(figsize=(14, 12))
_gs = fig.add_gridspec(4, 4, height_ratios=[1.0, 1.3, 1.1, 1.1], hspace=0.55, wspace=0.25)

# ---- (A) full recording ------------------------------------------------------
_axA = fig.add_subplot(_gs[0, :])
_nA = min(int(6.0 * FS), len(_d["filt"])); _tA = np.arange(_nA) / FS
_axA.plot(_tA, _d["filt"][:_nA], color="#333", lw=0.6)
_pA = _d["pk"][_d["pk"] < _nA]
_axA.plot(_pA / FS, _d["filt"][_pA], "v", color="#c0392b", ms=5)
_axA.set_title(f"(A)  Full filtered recording — Animal {FIG6_A}, "
               f"{len(_d['pk'])} beats detected (HR {_d['hr']:.0f} bpm)",
               fontweight="bold", loc="left")
_axA.set_xlabel("time (s)"); _axA.set_ylabel("mV"); _axA.grid(alpha=.2)

# ---- (B) one averaged beat with landmarks -----------------------------------
_axB = fig.add_subplot(_gs[1, :])
_axB.plot(_d["t_ms"], _d["tmpl"], color="black", lw=2)
_axB.axhline(0, color="gray", ls=":", lw=1, alpha=.7)
for _nm, _lo, _hi, _md, _col in _LM:
    _tm = _landmark(_d["tmpl"], _d["t_ms"], _lo, _hi, _md)
    if _tm is None:
        continue
    _i = int(np.argmin(np.abs(_d["t_ms"] - _tm)))
    _axB.plot(_tm, _d["tmpl"][_i], "o", color=_col, ms=10, zorder=5)
    _axB.annotate(_nm, (_tm, _d["tmpl"][_i]), textcoords="offset points",
                  xytext=(0, 12), ha="center", fontweight="bold",
                  fontsize=12, color=_col)
_axB.set_xlim(-100, 110)
_axB.set_title("(B)  A single averaged beat, with the extracted landmarks "
               "(P, Q, R, S, J, inverted T)", fontweight="bold", loc="left")
_axB.set_xlabel("time relative to R-peak (ms)"); _axB.set_ylabel("mV")
_axB.grid(alpha=.2)

# ---- (C) every individual beat overlaid, per animal --------------------------
_rows = [("clean", FIG6_CLEAN, "#1f6f4a"), ("flagged", FIG6_FLAGGED, "#c0392b")]
_prepped = {a: _fig6_prep(a) for _, _ids, _ in _rows for a in _ids}
# y-limits PER ROW: clean and flagged differ by up to ~50x in beat-to-beat
# spread, so a single shared scale would flatten the clean animals to a line.
def _row_lims(ids):
    _b = [_prepped[a]["beats"] for a in ids
          if _prepped.get(a) and _prepped[a]["beats"] is not None]
    if not _b:
        return (-1, 1)
    _lo, _hi = np.percentile(np.concatenate(_b), [0.5, 99.5])
    _pad = 0.15 * (_hi - _lo)
    return (_lo - _pad, _hi + _pad)
_lims = {"clean": _row_lims(FIG6_CLEAN), "flagged": _row_lims(FIG6_FLAGGED)}

for _r, (_lab, _ids, _col) in enumerate(_rows):
    for _c2, _aid in enumerate(_ids):
        _ax = fig.add_subplot(_gs[2 + _r, _c2])
        _p = _prepped.get(_aid)
        if _p is None or _p["beats"] is None:
            _ax.text(.5, .5, f"A{_aid}\nno beats", ha="center", va="center",
                     transform=_ax.transAxes); _ax.set_xticks([]); _ax.set_yticks([]); continue
        _bt = _p["beats"]
        _sub = _bt[np.linspace(0, len(_bt) - 1, min(150, len(_bt))).astype(int)]
        for _b in _sub:
            _ax.plot(_p["t_ms"], _b, color="#888", lw=0.4, alpha=0.25)
        _ax.plot(_p["t_ms"], _p["tmpl"], color=_col, lw=1.8)
        _ax.set_ylim(*_lims[_lab]); _ax.set_xlim(-100, 110)
        _qw = (_p["t_ms"] >= -15) & (_p["t_ms"] <= 35)
        _sd = float(np.mean(np.std(_bt[:, _qw], axis=0)))     # beat-to-beat spread
        _ax.set_title("A%d  %d beats  rr_cv %.2f  SD %.3f mV"
                      % (_aid, len(_bt), _p["cv"], _sd),
                      fontsize=8, fontweight="bold", color=_col)
        _ax.grid(alpha=.2)
        if _c2 == 0:
            _ax.set_ylabel(f"{_lab.upper()}\nmV", fontweight="bold")
        else:
            _ax.set_yticklabels([])
        if _r == 1:
            _ax.set_xlabel("ms from R")

fig.suptitle("Beat-level structure: full recording, one averaged beat, and every "
             "individual beat overlaid\n(C) top row = clean animals (tight bundles); "
             "bottom row = flagged animals (scattered)  —  note the two rows use DIFFERENT y-scales",
             fontweight="bold", fontsize=13)
import figutil; figutil.explode(fig, "beat_structure", outdir=str(FIGURES_DIR))
plt.show()

print("beat-to-beat spread (SD across beats, averaged over the QRS window):")
for _lab, _ids, _ in _rows:
    for _aid in _ids:
        _p = _prepped.get(_aid)
        if _p is None or _p["beats"] is None:
            print(f"  {_lab:8s} A{_aid}: no beats"); continue
        _m = (_p["t_ms"] >= -15) & (_p["t_ms"] <= 35)
        _sd = float(np.mean(np.std(_p["beats"][:, _m], axis=0)))
        print(f"  {_lab:8s} A{_aid}: {_sd:.4f} mV   (n={len(_p['beats'])} beats, rr_cv {_p['cv']:.2f})")


## Feature detection on INDIVIDUAL beats (not the averaged template)

Averaging suppresses noise, so detecting landmarks on an averaged beat is an
easier problem than the real one -- and the averaged beat is a mathematical
construct that need not correspond to any heartbeat the animal actually had.

This cell therefore runs the pipeline's own `_measure_one` / `_qt_per_beat` on
**individual, unaveraged beats** and marks the landmarks on each one separately.
The per-beat measurements are then compared with the value obtained from the
averaged template, so the effect of averaging can be quantified rather than
assumed.


In [ ]:
PERBEAT_ANIMALS = [230, 208, 239]     # high QT yield (97%, 86%, 82%)
N_SHOW = 3                            # individual beats displayed per animal

_LM_POS = [("P", -70, -35, "max"), ("Q", -14, -2, "min"),
           ("S", 0, 8, "min"), ("J", 4, 16, "max"), ("T", 14, 30, "min")]

def _perbeat_report(aid, n_show=N_SHOW):
    _row = df[df.animal_id == aid].iloc[0]
    _p = DATA_DIR / str(_row["source_file"])
    _nh, _l, _c, _ = parse_header_and_layout(_p)
    _v, _mk = load_ecg_file(_p, _nh, _c)
    _s, _e = find_baseline_window(_mk, len(_v))[:2]
    _filt = bandpass_filter(_v[_s:_e])
    _pk, _inv, _hr, _ = detect_r_peaks(_filt)
    if _inv:
        _filt = -_filt
    _rr = np.diff(_pk) * (1000.0 / FS)
    _rrm = float(np.median(_rr))
    _tmpl, _t_ms, _beats = average_beats(_filt, _pk)
    _bm = _t_ms < -75                                     # far-baseline = noise region

    # every beat, measured with the pipeline's own functions
    _qt = np.array([_qt_per_beat(_b, _t_ms, _rrm) for _b in _beats], dtype=float)
    _mm = [_measure_one(_b, _t_ms) for _b in _beats]
    _ramp = np.array([m[0] for m in _mm], dtype=float)
    _qrs = np.array([m[1] for m in _mm], dtype=float)
    _ok = np.where(~np.isnan(_qt))[0]                     # beats where QT resolved

    # landmark position consistency across ALL beats
    _noise = float(np.mean([np.std(_b[_bm]) for _b in _beats]))
    _cons = []
    for _nm, _lo, _hi, _md in _LM_POS:
        _ps, _as = [], []
        for _b in _beats:
            _bb = float(np.median(_b[_bm]))
            _m = (_t_ms >= _lo) & (_t_ms <= _hi)
            _seg, _tt = (_b - _bb)[_m], _t_ms[_m]
            _i = int(np.argmin(_seg) if _md == "min" else np.argmax(_seg))
            _ps.append(_tt[_i]); _as.append(abs(_seg[_i]))
        _ps = np.array(_ps)
        _rand = (_hi - _lo) / np.sqrt(12)                 # SD if picking at random
        _cons.append(dict(wave=_nm, pos=_ps.mean(), sd=_ps.std(),
                          rand_sd=_rand, ratio=_ps.std() / _rand,
                          snr=np.mean(_as) / _noise))

    # ---- figure ----
    _pick = _ok[np.linspace(0, len(_ok) - 1, n_show + 2).astype(int)[1:-1]] if len(_ok) >= n_show + 2 \
            else _ok[:n_show]
    fig = plt.figure(figsize=(14, 8.5))
    _gs = fig.add_gridspec(2, 3, height_ratios=[1.15, 1.0], hspace=0.45, wspace=0.24)

    for _k, _bi in enumerate(_pick):
        _b = _beats[_bi]
        _bb = float(np.median(_b[_bm]))
        _y = _b - _bb
        _ax = fig.add_subplot(_gs[0, _k])
        _ax.plot(_t_ms, _y, color="#333", lw=1.4)
        _ax.axhline(0, color="gray", ls=":", lw=1, alpha=.7)
        for _nm, _lo, _hi, _md, _col in _LM:
            _tt = _landmark(_y, _t_ms, _lo, _hi, _md)
            if _tt is None:
                continue
            _j = int(np.argmin(np.abs(_t_ms - _tt)))
            _ax.plot(_tt, _y[_j], "o", color=_col, ms=8, zorder=5)
            _ax.annotate(_nm, (_tt, _y[_j]), textcoords="offset points",
                         xytext=(0, 9), ha="center", fontweight="bold", color=_col)
        _ax.set_xlim(-100, 110)
        _ax.set_title("beat #%d    QT %.1f ms    R %.2f mV"
                      % (_bi, _qt[_bi], _ramp[_bi]), fontsize=10, fontweight="bold")
        _ax.set_xlabel("ms from R"); _ax.grid(alpha=.2)
        if _k == 0:
            _ax.set_ylabel("mV")

    # QT histogram
    _ax = fig.add_subplot(_gs[1, 0])
    _v2 = _qt[~np.isnan(_qt)]
    _ax.hist(_v2, bins=25, color="#8fb9a8", edgecolor="#4a7c64")
    _ax.axvline(np.mean(_v2), color="#1f6f4a", lw=2, label="mean %.1f ms" % np.mean(_v2))
    _ax.axvline(np.median(_v2), color="#c0392b", lw=2, ls="--",
                label="median %.1f ms" % np.median(_v2))
    _ax.set_title("QT per beat  (%d of %d beats, %.0f%%)"
                  % (len(_v2), len(_qt), 100 * len(_v2) / len(_qt)),
                  fontsize=10, fontweight="bold")
    _ax.set_xlabel("ms"); _ax.set_ylabel("beats"); _ax.legend(fontsize=8); _ax.grid(alpha=.2)

    # R amplitude histogram
    _ax = fig.add_subplot(_gs[1, 1])
    _ax.hist(_ramp, bins=25, color="#9fb8d0", edgecolor="#4a6c8c")
    _ax.axvline(_ramp.mean(), color="#1f4f7a", lw=2, label="mean %.3f mV" % _ramp.mean())
    _ax.set_title("R amplitude per beat  (n=%d)" % len(_ramp), fontsize=10, fontweight="bold")
    _ax.set_xlabel("mV"); _ax.legend(fontsize=8); _ax.grid(alpha=.2)

    # landmark position consistency
    _ax = fig.add_subplot(_gs[1, 2]); _ax.axis("off")
    _txt = "landmark position across ALL %d beats\n\n" % len(_beats)
    _txt += "%-4s %8s %7s %8s\n" % ("wave", "mean", "SD", "vs rand")
    _txt += "-" * 32 + "\n"
    for _c3 in _cons:
        _flag = "ok" if _c3["ratio"] < 0.45 else ("~" if _c3["ratio"] < 0.75 else "NOISE")
        _txt += "%-4s %7.1f%s %6.2f %7.2f %s\n" % (
            _c3["wave"], _c3["pos"], "ms", _c3["sd"], _c3["ratio"], _flag)
    _txt += "\nSD << random SD  =>  real wave,\nnot noise.  noise floor %.4f mV" % _noise
    _ax.text(0.0, 0.98, _txt, family="monospace", fontsize=8.5, va="top")

    fig.suptitle("Feature detection on INDIVIDUAL beats  -  Animal %d, %d beats "
                 "(landmarks found on each beat separately)" % (aid, len(_beats)),
                 fontweight="bold", fontsize=12.5)
    import figutil; figutil.explode(fig, "perbeat_A%d" % aid, outdir=str(FIGURES_DIR))
    plt.show(); plt.close(fig)

    print("A%d: %d beats | QT on %d (%.0f%%), mean %.1f median %.1f ms | R %.3f mV"
          % (aid, len(_beats), len(_v2), 100 * len(_v2) / len(_qt),
             np.mean(_v2), np.median(_v2), _ramp.mean()))
    for _c3 in _cons:
        print("    %-2s pos %6.1f ms  SD %.2f  (random %.2f)  amp/noise %.1fx"
              % (_c3["wave"], _c3["pos"], _c3["sd"], _c3["rand_sd"], _c3["snr"]))
    return _cons

for _a in PERBEAT_ANIMALS:
    _perbeat_report(_a)
    print()
